In [ ]:
"""
====================================================================================================
DECISION INTELLIGENCE FOR RESILIENT REVERSE LOGISTICS
A Three-Phase Matheuristic Framework for the MP-HF-SD-2E-LIRP
====================================================================================================
Description:
This script implements a comprehensive Decision Intelligence pipeline to solve the Multi-Period 
Heterogeneous Fleet Split Delivery Two-Echelon Location-Inventory-Routing Problem (MP-HF-SD-2E-LIRP).
It minimizes the 15-year Life Cycle Cost (LCC) of an agricultural reverse logistics network 
under stochastic climate-induced supply volatility. 

The framework bridges Artificial Intelligence (Unsupervised Learning) and Operations Research 
(Metaheuristics and Exact Mathematical Programming) through a highly decoupled three-phase architecture:

----------------------------------------------------------------------------------------------------
PIPELINE ARCHITECTURE:
----------------------------------------------------------------------------------------------------
[INIT] AI-Driven Structural-Start:
       Uses a capacity-driven, demand-weighted K-Means clustering algorithm to map the theoretical 
       extremes of the efficiency frontier. This seeds the metaheuristic population,
       preventing cold-start inefficiencies.

[PHASE 1] Strategic Network Design (Metaheuristics):
       It evaluates candidate facility locations and capacities using a fast, low-fidelity 
       surrogate evaluator (Order-First Split-Second heuristic + Analytical Penalty Proxies).

[PHASE 2] Tactical 2E-IRP (Heuristic Column Generation + + Set Partitioning MILP):
       With the spatial topology locked by Phase 1, Phase 2 generates an 'a priori' column pool 
       of routes using a Randomized Clarke-Wright algorithm polished by a Bounded 2-Opt. 
       Finally, an exact Set Partitioning MILP (via Gurobi) endogenously sizes the fleet and
       synchronizes multi-echelon inventory buffering.

[PHASE 3] Operational Fleet Scheduling (Assignment MILP):
       Unrolls the aggregate frequencies from Phase 2 into chronological, physical vehicle 
       assignments. Independent MILPs (per period and fleet class) balance driver workloads, 
       minimize exact labor overtime, and penalize deadheading friction.

----------------------------------------------------------------------------------------------------
Outputs:
- Detailed financial LCC breakdown (CAPEX, OPEX, Transport, Inventory, and Penalties).
- Geospatial mapping of the active reverse logistics network and vehicle routing.
- Comprehensive Excel spreadsheets with multi-echelon mass balances and operational reporting.

Requirements:
Python 3.10+, Pyomo, Gurobipy (with valid WLS/Academic license), Scikit-Learn, Pandas, 
Numpy, Geopandas, Matplotlib, Seaborn.
====================================================================================================
"""

In [ ]:
import os
import gc  # PARA LIMPEZA DE MEMÓRIA

# Caminho absoluto para o seu SSD no disco F
node_dir = 'F:/gurobi_nodes'

# Criar pasta para os Nodefiles do Gurobi no SSD
if not os.path.exists(node_dir):
    os.makedirs(node_dir)

import random
import math
import copy
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.patches import Patch
from sklearn.cluster import KMeans
import seaborn as sns
import geopandas as gpd
import pyomo.environ as pyo
from openpyxl.drawing.image import Image as XLImage
import unicodedata
import gurobipy as gp
import time
from matplotlib import ticker
import matplotlib.lines as mlines

# Configurações visuais
sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)
plt.rcParams['font.family'] = 'serif'

In [ ]:
# ==============================================================================
# CONFIGURAÇÃO DA LICENÇA GUROBI WLS
# ==============================================================================
print(f"\n{'='*60}\n>> Configuring Gurobi WLS license...\n{'='*60}\n")
WLS_ACCESS_ID = "INSERT HERE" # Replace with your actual WLS access ID
WLS_SECRET    = "INSERT HERE" # Replace with your actual WLS secret
LICENSE_ID    = 1234567 # Replace with your actual license ID

def setup_gurobi():
    lic_content = f"WLSACCESSID={WLS_ACCESS_ID}\nWLSSECRET={WLS_SECRET}\nLICENSEID={LICENSE_ID}\n"

    home_dir = os.path.expanduser("~")
    lic_path = os.path.join(home_dir, "gurobi.lic")

    with open(lic_path, "w") as f:
        f.write(lic_content)

    os.environ["GRB_LICENSE_FILE"] = lic_path

    try:
        env = gp.Env(empty=True)
        env.setParam('OutputFlag', 0) # Desativa a saída do Gurobi
        env.setParam('WLSACCESSID', WLS_ACCESS_ID)
        env.setParam('WLSSECRET', WLS_SECRET)
        env.setParam('LICENSEID', LICENSE_ID)
        env.start()
        env.dispose()
        # print(f">> Gurobi WLS Ativado! Licença salva em: {lic_path}")
        return True
    except Exception as e:
        print(f">> [!] Erro na licença: {e}")
        return False

HAS_GUROBI = setup_gurobi()

# ==============================================================================
# 1. CLASSE DE DADOS GLOBAIS
# ==============================================================================
class LRPData:
    def __init__(self, instance, cenario_base):
        print(f"\n--- Carregando dados para OTIMIZAÇÃO 2E-LIRP (Estado Completo) ---")

        def normalize_txt(text):
            if not isinstance(text, str): return str(text)
            return unicodedata.normalize('NFKD', text).encode('ASCII', 'ignore').decode('ASCII').upper().strip()

        self.instance = instance

        df_mensal = pd.read_excel(self.instance, sheet_name='Mensal')
        df_mensal['Município_Norm'] = df_mensal['Município'].apply(normalize_txt)
        meses_cols =["Janeiro", "Fevereiro", "Março", "Abril", "Maio", "Junho", "Julho", "Agosto", "Setembro", "Outubro", "Novembro", "Dezembro"]

        self.df_cluster = df_mensal.copy().reset_index(drop=True)
        self.num_clientes = len(self.df_cluster)
        self.num_candidatos = self.num_clientes

        df_costs = pd.read_excel(self.instance, sheet_name='Costs', index_col=0)
        ids_cluster = self.df_cluster['ID'].values
        df_costs.index = df_costs.index.astype(type(ids_cluster[0]))
        df_costs.columns = df_costs.columns.astype(type(ids_cluster[0]))
        self.dist_matrix = df_costs.loc[ids_cluster, ids_cluster].values
        self.dist_matrix = np.maximum(self.dist_matrix, self.dist_matrix.T)

        path_shp = "TO_Municipios_2022.json"
        if not os.path.exists(path_shp):
            import requests
            req = requests.get("https://raw.githubusercontent.com/tbrugz/geodata-br/master/geojson/geojs-17-mun.json")
            with open(path_shp, 'w', encoding='utf-8') as f: f.write(req.text)

        self.gdf_estado = gpd.read_file(path_shp)
        self.gdf_estado['name_upper'] = self.gdf_estado['name'].apply(normalize_txt)
        self.df_cluster['model_idx'] = self.df_cluster.index
        self.gdf_estado = self.gdf_estado.merge(self.df_cluster[['model_idx', 'Município_Norm']], left_on='name_upper', right_on='Município_Norm', how='left')

        self.horizonte_anos = 15

        self.cenario_base = cenario_base # Opções: "Otimista", "Realista", "Pessimista"

        print(f"\n{'='*60}\nCenário: {self.cenario_base}\n{'='*60}")

        # Taxa de Desconto (r) = Custo de Capital da empresa (SELIC Jan 2010 a Jan 2026)

        if self.cenario_base == "Otimista":
            self.taxa_desconto = 0.0714  #  7.14% P25 (Inflação controlada, crescimento econômico estável, ambiente de negócios favorável)

        if self.cenario_base == "Realista":
            self.taxa_desconto = 0.1065  # 10.65% P50 (Inflação moderada, crescimento econômico alinhado com expectativas, ambiente de negócios competitivo)

        if self.cenario_base == "Pessimista":
            self.taxa_desconto = 0.1315  # 13.15% P75 (Inflação alta, crescimento econômico fraco, ambiente de negócios desfavorável)

        # =====================================================================
        # PROJEÇÕES DE CRESCIMENTO E FATORES DE PICO PARA LCC
        # =====================================================================

        '''
        # Versão Regional Norte 6.0 (Com Curva S Logistic e Simulação de Monte Carlo para projeção de demanda e consumo de agrotóxicos):

            --- Optimistic ---
            Multiplicadores E[Φ] (15 anos):
            [1.0094, 1.0649, 1.1181, 1.1679, 1.2182, 1.2684, 1.3152, 1.3578, 1.3999, 1.4442, 1.4821, 1.5221, 1.5532, 1.5905, 1.6235]
            Multiplicadores P95 (15 anos):
            [1.1851, 1.2488, 1.3086, 1.3716, 1.4234, 1.4861, 1.5430, 1.5913, 1.6413, 1.6933, 1.7461, 1.7881, 1.8205, 1.8632, 1.9076]

            --- Realistic ---
            Multiplicadores E[Φ] (15 anos):
            [1.0849, 1.1532, 1.2191, 1.2807, 1.3441, 1.4092, 1.4699, 1.5249, 1.5805, 1.6415, 1.6930, 1.7489, 1.7909, 1.8450, 1.8928]
            Multiplicadores P95 (15 anos):
            [1.3748, 1.4590, 1.5378, 1.6237, 1.6914, 1.7805, 1.8608, 1.9276, 1.9991, 2.0761, 2.1565, 2.2184, 2.2642, 2.3307, 2.4019]

            --- Pessimistic ---
            Multiplicadores E[Φ] (15 anos):
            [1.2265, 1.3189, 1.4088, 1.4922, 1.5799, 1.6743, 1.7609, 1.8387, 1.9189, 2.0124, 2.0900, 2.1762, 2.2362, 2.3232, 2.3994]
            Multiplicadores P95 (15 anos):
            [1.7968, 1.9279, 2.0491, 2.1884, 2.2894, 2.4413, 2.5764, 2.6829, 2.8034, 2.9398, 3.0874, 3.1940, 3.2644, 3.3870, 3.5238]

        '''
        if self.cenario_base == "Otimista":

            # 1. LISTA DA MÉDIA ESPERADA (Para dimensionamento do Fator de Crescimento do LCC)
            self.curva_crescimento_demanda =[1.0094, 1.0649, 1.1181, 1.1679, 1.2182, 1.2684, 1.3152, 1.3578, 1.3999, 1.4442, 1.4821, 1.5221, 1.5532, 1.5905, 1.6235] # Crescimento Esperado da demanda anual (Versão: Regional Norte 6.0)
            self.crescimento_pico = 1.6235 # Valor de crescimento esperado da demanda no ano 15 (Versão: Regional Norte 6.0)

            # 2. O FATOR DE PICO ROBUSTO (P95) (Usado para dimensionamento dos galpões e avaliação de penalidades por pico)
            self.fator_pico_serie = [1.1851, 1.2488, 1.3086, 1.3716, 1.4234, 1.4861, 1.5430, 1.5913, 1.6413, 1.6933, 1.7461, 1.7881, 1.8205, 1.8632, 1.9076] # Fator de Pico P95 para cada ano (Versão: Regional Norte 6.0)
            self.fator_pico = 1.9076 # Fator de Pico P95 para o ano 15 (Versão: Regional Norte 6.0)

        if self.cenario_base == "Realista":

            self.curva_crescimento_demanda =[1.0849, 1.1532, 1.2191, 1.2807, 1.3441, 1.4092, 1.4699, 1.5249, 1.5805, 1.6415, 1.6930, 1.7489, 1.7909, 1.8450, 1.8928]
            self.crescimento_pico = 1.8928

            self.fator_pico_serie = [1.3748, 1.4590, 1.5378, 1.6237, 1.6914, 1.7805, 1.8608, 1.9276, 1.9991, 2.0761, 2.1565, 2.2184, 2.2642, 2.3307, 2.4019]
            self.fator_pico = 2.4019

        if self.cenario_base == "Pessimista":

            self.curva_crescimento_demanda =[1.2265, 1.3189, 1.4088, 1.4922, 1.5799, 1.6743, 1.7609, 1.8387, 1.9189, 2.0124, 2.0900, 2.1762, 2.2362, 2.3232, 2.3994]
            self.crescimento_pico = 2.3994

            self.fator_pico_serie = [1.7968, 1.9279, 2.0491, 2.1884, 2.2894, 2.4413, 2.5764, 2.6829, 2.8034, 2.9398, 3.0874, 3.1940, 3.2644, 3.3870, 3.5238]
            self.fator_pico = 3.5238

        # =====================================================================
        # MODELO LOGÍSTICO DE CRESCIMENTO E MATEMÁTICA FINANCEIRA INFLACIONÁRIA
        # ===================================================================== 

        if self.cenario_base == "Otimista":
            
            # Taxas de Inflação / Escalada de Custos (Cenário Brasileiro com Margem de Segurança)
            self.i_opex   = 0.0408    # Inflação sobre infraestrutura (INCC-M FGV) P25 = 0.0408, P50 = 0.0634, P75 = 0.0794
            self.i_fleet  = 0.0300    # Inflação sobre aquisição de caminhões pesados (IPP IBGE) P25 = 0.03, P50 = 0.05, P75 = 0.09
            self.i_transp = (0.6 * 0.000 + 0.4 * 0.0247)
                                     # Inflação sobre diesel e custos variáveis de transporte:
                                     # Diesel (ANP):   P25 = 0.000, P50 = 0.040, P75 = 0.120     Peso: 0.60 (Martino et al., 2009)
                                     # Outros (IGP-M): P25 = 0.0247, P50 = 0.0717, P75 = 0.0904  Peso: 0.40 (Martino et al., 2009)

        if self.cenario_base == "Realista":

            # Taxas de Inflação / Escalada de Custos (Cenário Brasileiro com Margem de Segurança)
            self.i_opex   = 0.0634    # Inflação sobre infraestrutura (INCC-M FGV) P25 = 0.0408, P50 = 0.0634, P75 = 0.0794
            self.i_fleet  = 0.0500    # Inflação sobre aquisição de caminhões pesados (IPP IBGE) P25 = 0.03, P50 = 0.05, P75 = 0.09
            self.i_transp = (0.6 * 0.040 + 0.4 * 0.0717)
                                     # Inflação sobre diesel e custos variáveis de transporte:
                                     # Diesel (ANP):   P25 = 0.000, P50 = 0.040, P75 = 0.120     Peso: 0.60 (Martino et al., 2009)
                                     # Outros (IGP-M): P25 = 0.0247, P50 = 0.0717, P75 = 0.0904  Peso: 0.40 (Martino et al., 2009)

        if self.cenario_base == "Pessimista":

            # Taxas de Inflação / Escalada de Custos (Cenário Brasileiro com Margem de Segurança)
            self.i_opex   = 0.0794    # Inflação sobre infraestrutura (INCC-M FGV) P25 = 0.0408, P50 = 0.0634, P75 = 0.0794
            self.i_fleet  = 0.0900    # Inflação sobre aquisição de caminhões pesados (IPP IBGE) P25 = 0.03, P50 = 0.05, P75 = 0.09
            self.i_transp = (0.6 * 0.120 + 0.4 * 0.0904)
                                     # Inflação sobre diesel e custos variáveis de transporte:
                                     # Diesel (ANP):   P25 = 0.000, P50 = 0.040, P75 = 0.120     Peso: 0.60 (Martino et al., 2009)
                                     # Outros (IGP-M): P25 = 0.0247, P50 = 0.0717, P75 = 0.0904  Peso: 0.40 (Martino et al., 2009)
        
        # 1. Lambda OPEX (Aplica inflação imobiliária e WACC)
        self.factor_opex = sum([((1 + self.i_opex)**n) / ((1 + self.taxa_desconto)**n) for n in range(1, self.horizonte_anos + 1)])
        
        # 2. Lambda Fleet (Aplica inflação automotiva e WACC)
        self.factor_fleet = sum([((1 + self.i_fleet)**n) / ((1 + self.taxa_desconto)**n) for n in range(1, self.horizonte_anos + 1)])
        
        # 3. Lambda Var (Aplica inflação de transportes, deflator de volume da Curva-S e WACC)
        self.factor_lcc = sum([
            (self.curva_crescimento_demanda[n-1] / self.crescimento_pico) * (((1 + self.i_transp)**n) / ((1 + self.taxa_desconto)**n))
            for n in range(1, self.horizonte_anos + 1)
        ])
        # =====================================================================

        self.horas_trabalho = 240        # Horas de trabalho disponíveis por veículo por mês (ajustável)
        self.max_rotas = 20              # Limite de rotas por depósito para evitar explosão combinatória na Fase 1
        self.tempo_servico = 2.5         # Tempo médio de serviço por cliente (horas)
        self.nomes_periodos = meses_cols # Mantém os nomes dos meses para referência, mas a modelagem é feita em períodos genéricos (0...11)
        self.num_periodos = 12           # Número de períodos (meses) para modelagem

        self.coords = {}
        for idx, row in self.df_cluster.iterrows():
            pos = (row['Longitude'], row['Latitude'])
            self.coords[f'Cli_{idx}'] = pos
            self.coords[f'Dep_{idx}'] = pos

        self.demanda_base = self.df_cluster[meses_cols].values  # Não há mais ajuste de fator de geração de resíduos
        self.demanda_pico = self.demanda_base * self.fator_pico # Pico de demanda ajustado para dimensionamento e penalidades (P95)
        self.cap_estoque_cliente = {i: max(1.2 * np.max(self.demanda_pico[i, :]), 100.0) for i in range(self.num_clientes)}

        self.custo_visita = 60.0        # Custo fixo por visita a cliente (tempo de serviço, manuseio, etc.)
        self.custo_estoque_cli = 0.015  # Custo de manter estoque no cliente (mais caro que no depósito)
        self.custo_estoque_dep = 0.010  # Custo de manter estoque no depósito (mais barato que no cliente)
        self.gamma_hora_extra = 1.5     # Multiplicador de custo para horas extras (assume 50% mais caro que hora normal)

        # =====================================================================
        # PARÂMETROS DE PENALIDADE E NÃO-CONFORMIDADE
        # =====================================================================
        # 1. Penalidades Matemáticas (Big-M) - Guiam o Solver e Metaheurísticas
        self.big_m_dev   = 1.5      # Multa solver: Desvio da Indústria (R$/kg)
        self.big_m_lost  = 1000.0   # Multa solver: Transbordo na Fazenda (R$/kg)
        self.big_m_over  = 1000.0   # Multa solver: Gargalo no Armazém (R$/kg)
        self.big_m_spot  = 9000.0   # Multa solver: Carreta Spot E2 (R$/veículo)

        # 2. Custos Financeiros Reais (Shadow Prices) - Para Planilha e Gráfico
        self.custo_real_dev   = 0.50    # Desconto no valor do material por quebra de SLA (R$/kg)
        self.custo_real_lost  = 5.00    # Custo de frete emergencial (R$/kg)
        self.custo_real_over  = 2.00    # Custo de aluguel de galpão extra (R$/kg)
        self.custo_real_spot  = 5000.0  # Ágio real cobrado no mercado Spot por carreta (R$/veículo)
        # =====================================================================

        # --- PARÂMETROS DO ESCALÃO 2 (INDÚSTRIA) ---
        self.cap_carreta_industria = 25000
        self.fluxo_mensal_ideal = np.sum(self.demanda_pico) / 12.0
        viagens_mensais_medias = self.fluxo_mensal_ideal / self.cap_carreta_industria
        self.max_carretas_industria_mes = math.ceil(viagens_mensais_medias * 1.5)

        self.veiculos_info =[
            {'id': 'VUC',     'idx': 0, 'cap': 4000,  'fixo': 11000, 'var': 2.50, 'vel': 60, 'hora': 60},
            {'id': 'Truck',   'idx': 1, 'cap': 12000, 'fixo': 17000, 'var': 3.50, 'vel': 55, 'hora': 90},
            {'id': 'Carreta', 'idx': 2, 'cap': 25000, 'fixo': 41000, 'var': 5.50, 'vel': 50, 'hora': 120}
        ]
        
        self.tipos_veiculo =[v['id'] for v in self.veiculos_info]

        self.niveis_cap_info =[
            {'nome': 'Closed',  'cap': 0,         'capex': 0,       'opex': 0},
            {'nome': 'Pequeno', 'cap': 130000/12, 'capex': 3500000, 'opex': 30000},
            {'nome': 'Médio',   'cap': 390000/12, 'capex': 6000000, 'opex': 45000},
            {'nome': 'Grande',  'cap': 975000/12, 'capex': 7600000, 'opex': 70000}
        ]

    def get_distancia(self, i, j):
        d = float(self.dist_matrix[i, j])
        return 5.0 if d < 1.0 else d

    def get_baseline_depots(self):
        base_dep = [0] * self.num_clientes
        self.proxy_distances = {} # Armazena a distância extra para o Inpev Real
        self.baseline_config = None
        
        postos = ['ARAGUAINA', 'GURUPI', 'LAGOA DA CONFUSAO']
        centrais = ['CAMPOS LINDOS', 'CARIRI DO TOCANTINS', 'PEDRO AFONSO', 'SILVANOPOLIS']
        
        # --- CORREÇÃO: Conjunto para evitar duplicar galpões Inpev em cidades divididas (Polos)
        inpev_alocados = set()

        # 1. Tenta alocar as instalações reais caso elas existam na matriz
        for idx, row in self.df_cluster.iterrows():
            n_raw = row['Município_Norm']
            
            # Limpa o sufixo artificial para buscar o nome raiz (Ex: "ARAGUAINA (POLO A)" -> "ARAGUAINA")
            n_base = n_raw.split('(')[0].strip()
            
            # Aloca o posto apenas se ainda não foi alocado (garante que só abrirá no Polo A)
            if n_base in postos and n_base not in inpev_alocados:
                base_dep[idx] = 1
                inpev_alocados.add(n_base)
            elif n_base in centrais and n_base not in inpev_alocados:
                base_dep[idx] = 2
                inpev_alocados.add(n_base)
            
        # 2. Tratamento para Micro-regiões (Sem instalações oficiais)
        if sum(base_dep) == 0:
            
            print(f"\n{'='*70}")
            print("🔀 CONFIGURAÇÃO PROXY BASELINE")
            print(f"{'='*70}\n")

            local_cities = []
            
            try:
                df_full_mensal = pd.read_excel('Dist_Full.xlsx', sheet_name='Mensal') 
                df_full_dist = pd.read_excel('Dist_Full.xlsx', sheet_name='Costs', index_col=0)
                
                df_full_dist.index = df_full_dist.index.astype(str)
                df_full_dist.columns = df_full_dist.columns.astype(str)
                df_full_mensal['ID'] = df_full_mensal['ID'].astype(str)
                
                def norm_txt(t): return unicodedata.normalize('NFKD', str(t)).encode('ASCII', 'ignore').decode('ASCII').upper().strip()
                df_full_mensal['Município_Norm'] = df_full_mensal['Município'].apply(norm_txt)
                
                inpev_ids = {}
                for _, row in df_full_mensal.iterrows():
                    n = row['Município_Norm']
                    if n in postos: inpev_ids[row['ID']] = {'nome': n, 'tipo': 1}
                    elif n in centrais: inpev_ids[row['ID']] = {'nome': n, 'tipo': 2}
                
                for idx, row in self.df_cluster.iterrows():
                    local_id = str(row['ID']).split("_")[0] 
                    min_dist = float('inf')
                    nearest_name, nearest_tipo = None, 1
                    
                    for inv_id, inv_info in inpev_ids.items():
                        try:
                            dist = float(df_full_dist.loc[local_id, inv_id])
                            if dist < min_dist:
                                min_dist, nearest_name, nearest_tipo = dist, inv_info['nome'], inv_info['tipo']
                        except KeyError: pass
                            
                    local_cities.append({'idx': idx, 'mun': row['Município_Norm'], 'dist': min_dist, 'nearest': nearest_name, 'tipo': nearest_tipo})
                    
            except Exception as e:
                print(f"⛔   [Aviso] Falha na Matriz Completa ({e}).")
                # (Mantenha seu bloco fallback de Haversine aqui se desejar)
                pass

            local_cities.sort(key=lambda x: x['dist'])
            demanda_total_regiao = np.sum(self.demanda_pico)
            capacidade_aberta = 0
            
            for city in local_cities:
                if capacidade_aberta >= demanda_total_regiao: break 
                    
                idx, nivel = city['idx'], city['tipo']
                base_dep[idx] = nivel
                
                # Armazena a distância de "Punição" para o Inpev Real!
                self.proxy_distances[idx] = city['dist']
                
                capacidade_aberta += self.niveis_cap_info[nivel]['cap'] * 12 
                print(f"🌐   -> Proxy Hub Nível {nivel} ativado em {city['mun']}")
                print(f"📍   -> Conexão rodoviária com {city['nearest']}: +{city['dist']:.1f} km)")

        self.baseline_config = list(base_dep) # Marca a assinatura do Baseline
        return base_dep

# ==============================================================================
# 2. FASE 1: AVALIADOR SURROGATE (ALGORITMO SPLIT DE PRINS COM SUPORTE A SPLIT DELIVERY)
# ==============================================================================
class Phase1Evaluator:
    def __init__(self, data):
        self.data = data
        self.demanda_media_cli = np.sum(self.data.demanda_pico, axis=1) / 12.0
        self.c_var_avg = np.mean([v['var'] for v in self.data.veiculos_info])
        self.cap_media_veiculo = 12000.0

    def evaluate(self, depot_config, fleet_config):
        
        # --- NOVO: CHECAGEM DE PROXY BASELINE ---
        is_baseline = hasattr(self.data, 'baseline_config') and depot_config == self.data.baseline_config
        
        def get_dist(a, b):
            d = self.data.get_distancia(a, b)
            # Se for o Baseline, soma a distância do Proxy Hub para o Inpev Externo
            if is_baseline:
                if a in self.data.proxy_distances: d += self.data.proxy_distances[a]
                if b in self.data.proxy_distances: d += self.data.proxy_distances[b]
            return d
        # ----------------------------------------

        capex = opex = fleet_fix = 0
        open_depots =[]

        # 1. Avalia Custos Fixos
        for i, lvl in enumerate(depot_config):
            if lvl > 0:
                capex += self.data.niveis_cap_info[lvl]['capex'] # CAPEX não tem desconto (t=0)
                # OPEX Anual * Fator de Anuidade
                opex += self.data.niveis_cap_info[lvl]['opex'] * 12 * self.data.factor_opex
                open_depots.append((i, self.data.niveis_cap_info[lvl]['cap']))

        if not open_depots: return float('inf')

        # 2. Custos da Frota (Custo Fixo Anual * Fator de Anuidade)
        for k, qtd in enumerate(fleet_config):
            fleet_fix += self.data.veiculos_info[k]['fixo'] * 12 * self.data.factor_fleet * qtd

        clusters = {d_idx:[] for d_idx, cap in open_depots}
        depot_loads = {d_idx: 0.0 for d_idx, cap in open_depots}

        # Clusterização (Nearest Depot)
        for c_idx in range(self.data.num_clientes):
            demanda = self.demanda_media_cli[c_idx]
            if demanda > 0.1:
                best_depot = min([d[0] for d in open_depots], key=lambda d: get_dist(c_idx, d))
                clusters[best_depot].append(c_idx)
                depot_loads[best_depot] += demanda

        var_cost_mensal = 0

        # --- ALGORITMO SPLIT DE PRINS (Com Suporte a Split Delivery) ---
        for d_idx, cap in open_depots:
            clientes_cluster = clusters[d_idx]
            if not clientes_cluster: continue
            
            # 1. Order-First: Cria um Giant Tour via Nearest Neighbor
            unvisited = set(clientes_cluster)
            curr = d_idx
            giant_tour =[]
            while unvisited:
                nxt = min(unvisited, key=lambda x: get_dist(curr, x))
                giant_tour.append(nxt)
                unvisited.remove(nxt)
                curr = nxt
                
            # 1.5. Refinamento Topológico (Bounded Fast 2-opt)
            # Remove cruzamentos grosseiros do NN antes do Split
            improved = True
            max_2opt_iters = 3 # Limite rigoroso para não travar a Fase 1
            iters = 0
            
            while improved and iters < max_2opt_iters:
                improved = False
                # Varre a rota procurando inversões que economizem distância
                for i in range(len(giant_tour) - 1):
                    for j in range(i + 2, len(giant_tour)):
                        n_i = giant_tour[i]
                        n_prev = giant_tour[i-1] if i > 0 else d_idx
                        n_j = giant_tour[j]
                        n_next = giant_tour[j+1] if j < len(giant_tour)-1 else d_idx

                        # Distância Atual
                        d_curr = get_dist(n_prev, n_i) + get_dist(n_j, n_next)
                        # Distância se invertermos o trecho [i...j]
                        d_new = get_dist(n_prev, n_j) + get_dist(n_i, n_next)

                        # Se a troca economiza distância (com margem para erro de ponto flutuante)
                        if d_new < d_curr - 1e-4:
                            giant_tour[i:j+1] = giant_tour[i:j+1][::-1] # Inverte o trecho in-place
                            improved = True
                iters += 1
                
            # 2. Split-Second: Programação Dinâmica
            n = len(giant_tour)
            V =[float('inf')] * (n + 1)
            V[0] = 0.0
            
            for i in range(n):
                if V[i] == float('inf'): continue
                load = 0.0
                dist = 0.0
                
                for j in range(i + 1, n + 1):
                    c = giant_tour[j - 1]
                    load += self.demanda_media_cli[c]
                    
                    if load > self.cap_media_veiculo:
                        # Trata clientes gigantescos (Split Delivery)
                        if j == i + 1:
                            # Se é o único cliente na rota e já estourou o caminhão, 
                            # faz múltiplas viagens de ida e volta
                            viagens = math.ceil(load / self.cap_media_veiculo)
                            dist_ida_volta = get_dist(d_idx, c) + get_dist(c, d_idx)
                            dist_total = dist_ida_volta * viagens
                            cost = (dist_total * self.c_var_avg) + (viagens * self.data.custo_visita)
                            
                            if V[i] + cost < V[j]:
                                V[j] = V[i] + cost
                        
                        break # O caminhão estourou, encerra a expansão DESSA rota e tenta o próximo fatiamento
                        
                    # Caso normal (cabe tudo no caminhão)
                    if j == i + 1:
                        # Rota direta: Ida e Volta
                        dist += get_dist(d_idx, c) + get_dist(c, d_idx)
                    else:
                        # Remove a volta ao depósito do nó anterior e adiciona o trecho direto + nova volta
                        prev_c = giant_tour[j - 2]
                        dist = dist - get_dist(prev_c, d_idx) + get_dist(prev_c, c) + get_dist(c, d_idx)
                        
                    cost = (dist * self.c_var_avg) + ((j - i) * self.data.custo_visita)
                    
                    # Atualiza o vetor de menor custo de fatiamento
                    if V[i] + cost < V[j]:
                        V[j] = V[i] + cost
                        
            # Adiciona o custo do fatiamento ótimo do cluster à fatura mensal
            var_cost_mensal += V[n]

        var_cost_lcc = var_cost_mensal * 12 * self.data.factor_lcc

        # =====================================================================
        # PENALIDADES
        # =====================================================================
        penalty = 0
        
        # 1. Buffered Peak (Pico Amortecido pelo IRP)
        # Avalia o pico residual de demanda de cada armazém após amortecer com a capacidade de estoque das fazendas (Split Delivery)
        for d_idx, cap in open_depots:
            clientes_cluster = clusters[d_idx]
            if not clientes_cluster: continue
            
            # 1.1 Avalia o mês de maior demanda do cluster (pico bruto)
            demanda_cluster_meses = np.sum(self.data.demanda_pico[clientes_cluster, :], axis=0)
            pico_cluster = np.max(demanda_cluster_meses)
            
            # 1.2 Soma o tamanho dos tanques de todas as fazendas vinculadas a esse armazém
            estoque_fazendas_cluster = sum(self.data.cap_estoque_cliente[c] for c in clientes_cluster)
            
            # 1.3 Assume que o MILP usará 70% dessa capacidade como "pulmão" para amortecer o pico (Vartheta Sensitivity Analysis)
            pico_amortecido = max(0, pico_cluster - (estoque_fazendas_cluster * 0.70))
            
            # 1.4 Se o Surto Residual estourar os 150% do armazém, o algoritmo é multado
            if pico_amortecido > (1.5 * cap):
                excesso_armazem = pico_amortecido - (1.5 * cap)
                # Inclui o factor_lcc para alinhar a magnitude da multa com os custos logísticos
                penalty += excesso_armazem * self.data.big_m_over * 3 * self.data.factor_lcc

        # 2. Insuficiência de Capacidade de Frota
        cap_frota_mensal = sum(fleet_config[k] * self.data.veiculos_info[k]['cap'] * (self.data.horas_trabalho/4) for k in range(3))
        demanda_total = sum(self.demanda_media_cli)
        if cap_frota_mensal < demanda_total:
            # Inclui o factor_lcc aqui também
            penalty += (demanda_total - cap_frota_mensal) * self.data.big_m_lost * 12 * self.data.factor_lcc

        # 3. Proxy de Estoque (Assume 50% do tempo no cliente, 50% no armazém)
        demanda_total_media = sum(self.demanda_media_cli)
        custo_estoque_mensal = (demanda_total_media * 0.5 * self.data.custo_estoque_cli) + \
                               (demanda_total_media * 0.5 * self.data.custo_estoque_dep)
        custo_estoque_lcc = custo_estoque_mensal * 12 * self.data.factor_lcc

        # 4. Proxy de Sazonalidade e Frota Spot E2 (Avalia o mês de Pico Máximo)
        demanda_mes_pico = np.max(np.sum(self.data.demanda_pico, axis=0))
        carretas_pico = math.ceil(demanda_mes_pico / self.data.cap_carreta_industria)
        multa_spot_estimada = 0
        if carretas_pico > self.data.max_carretas_industria_mes:
            # Multiplica pelos 3 meses prováveis de safra no ano
            multa_spot_estimada = (carretas_pico - self.data.max_carretas_industria_mes) * self.data.big_m_spot * 3 * self.data.factor_lcc

        # 5. Fator de Fricção (Deadheading e HE da Fase 3)
        var_cost_lcc = var_cost_lcc * 1.15 # Adiciona 15% de margem de erro operacional
        
        # O retorno agora fica perfeitamente alinhado com a proporção da Fase 2!
        return capex + opex + fleet_fix + var_cost_lcc + custo_estoque_lcc + multa_spot_estimada + penalty

# ==============================================================================
# 3. METAHEURÍSTICAS (FASE 1)
# ==============================================================================

# --------------------------------------------------------
# AI-DRIVEN STRUCTURAL-START (K-MEANS CLUSTERING)
# --------------------------------------------------------
def generate_ai_structural_start(data):
    """
    Gera arquétipos de redes logísticas utilizando Aprendizado Não-Supervisionado.
    """
    coords = data.df_cluster[['Latitude', 'Longitude']].values
    pesos = np.sum(data.demanda_pico, axis=1) / 12.0
    total_demand_p95 = np.sum(pesos) * data.fator_pico

    P_MAX = 10
    n_nodes = data.num_candidatos
    max_k_allowed = min(P_MAX, n_nodes)

    archetypes = []
    k_tested = set()

    niveis_validos = [info for info in data.niveis_cap_info if info['cap'] > 0]
    niveis_validos.sort(key=lambda x: x['cap'], reverse=True)

    for info in niveis_validos:
        cap_ref = info['cap']
        k_target = max(1, math.ceil(total_demand_p95 / cap_ref))
        k_target = min(k_target, max_k_allowed)

        if k_target not in k_tested:
            k_tested.add(k_target)
            kmeans = KMeans(n_clusters=k_target, random_state=42, n_init=10)
            labels = kmeans.fit_predict(coords, sample_weight=pesos)

            dep_arch = [0] * n_nodes
            for c in range(k_target):
                center = kmeans.cluster_centers_[c]
                dists = np.linalg.norm(coords - center, axis=1)
                best_idx = np.argmin(dists)

                cluster_demand = np.sum(pesos[labels == c])
                target_capacity = cluster_demand * data.fator_pico

                assigned_level = 1
                for lvl, cap_info in enumerate(data.niveis_cap_info):
                    if lvl == 0: continue
                    if cap_info['cap'] >= target_capacity:
                        assigned_level = lvl
                        break
                    assigned_level = lvl

                if dep_arch[best_idx] < assigned_level:
                    dep_arch[best_idx] = assigned_level

            archetypes.append(dep_arch)

    return archetypes

# --------------------------------------------------------
# SA: Simulated Annealing para otimização da rede logística (Fase 1)
# --------------------------------------------------------

def simulated_annealing(data, evaluator, max_time=200, max_evals=5000, p0_start=0.7, alpha=0.85, seed=42, use_ai_start=True, **kwargs):
    random.seed(seed); np.random.seed(seed)
    start_t = time.time(); evals = 0

    P_MAX = 10
    max_open_allowed = min(P_MAX, data.num_candidatos)

    # --- INICIALIZAÇÃO ---
    if use_ai_start:
        # Pool de Sementes de Elite: Baseline (Real ou Proxy) + Arquétipos da IA
        seeds_to_evaluate = []
        seeds_to_evaluate.append(data.get_baseline_depots())
        seeds_to_evaluate.extend(generate_ai_structural_start(data))
        
        best_seed_fit = float('inf')
        best_seed_dep = None
        best_seed_flt = [2, 3, 2] # Frota padrão para teste dos arquétipos

        for dep_arch in seeds_to_evaluate:
            # Proteção P_MAX (Trava de segurança)
            ativos = [i for i, x in enumerate(dep_arch) if x > 0]
            if len(ativos) > max_open_allowed:
                for i in random.sample(ativos, len(ativos) - max_open_allowed):
                    dep_arch[i] = 0
            if sum(dep_arch) == 0: dep_arch[0] = 2

            fit = evaluator.evaluate(dep_arch, best_seed_flt)
            evals += 1
            # O SA adota a Semente com o MENOR custo como seu estado inicial (Highest-fitness archetype)
            if fit < best_seed_fit:
                best_seed_fit = fit
                best_seed_dep = list(dep_arch)

        curr_dep = best_seed_dep
        curr_flt = best_seed_flt
        curr_fit = best_seed_fit
    else:
        # Modo Fatorial (Blind): Nasce cego, respeitando os limites
        curr_dep = [0] * data.num_candidatos
        min_open = min(2, max_open_allowed)
        num_open = random.randint(min_open, max_open_allowed)
        for idx in random.sample(range(data.num_candidatos), num_open):
            curr_dep[idx] = random.choice([1, 2, 3])
        if sum(curr_dep) == 0: curr_dep[0] = 2
        curr_flt = [random.randint(1, 5) for _ in range(3)]
        
        curr_fit = evaluator.evaluate(curr_dep, curr_flt); evals += 1

    best_fit = curr_fit; best_dep = list(curr_dep); best_flt = list(curr_flt)

    # --- BURN-IN PHASE (Average Delta T0) ---
    deltas = []
    temp_dep = list(curr_dep); temp_flt = list(curr_flt); temp_fit = curr_fit
    for _ in range(15):
        n_dep = list(temp_dep); n_flt = list(temp_flt)
        if random.random() < 0.5:
            n_dep[random.randint(0, data.num_candidatos-1)] = random.choice([0, 1, 2, 3])
        else:
            idx = random.randint(0, 2)
            n_flt[idx] = max(0, n_flt[idx] + random.choice([-1, 1]))
            if sum(n_flt) == 0: n_flt[0] = 1

        n_fit = evaluator.evaluate(n_dep, n_flt)
        if n_fit > temp_fit:
            deltas.append(n_fit - temp_fit)
        temp_dep, temp_flt, temp_fit = n_dep, n_flt, n_fit

    avg_delta = np.mean(deltas) if deltas else (best_fit * 0.05)
    T = -avg_delta / math.log(p0_start) if 0 < p0_start < 1 else avg_delta
    T_initial_locked = T

    history = [(evals, best_fit)]

    while (time.time() - start_t) < max_time and evals < max_evals:
        new_dep = list(curr_dep); new_flt = list(curr_flt)
        if random.random() < 0.5:
            new_dep[random.randint(0, data.num_candidatos-1)] = random.choice([0, 1, 2, 3])
        else:
            idx = random.randint(0, 2)
            new_flt[idx] = max(0, new_flt[idx] + random.choice([-1, 1]))
            if sum(new_flt) == 0: new_flt[0] = 1

        new_fit = evaluator.evaluate(new_dep, new_flt); evals += 1
        delta = new_fit - curr_fit

        if delta < 0 or (delta < 1e15 and random.random() < math.exp(-delta / max(T, 1e-10))):
            curr_dep, curr_flt, curr_fit = new_dep, new_flt, new_fit
            if curr_fit < best_fit:
                best_fit = curr_fit; best_dep = list(curr_dep); best_flt = list(curr_flt)

        T *= alpha

        # REHEATING
        if T < (avg_delta * 0.01):
            T = T_initial_locked * 0.5

        history.append((evals, best_fit))

        elapsed = time.time() - start_t
        if evals % 100 == 0:
            print(f"\r   [{'SA' if use_ai_start else 'SA'}]    Tempo: {elapsed:>5.1f}s / {max_time}s | Evals: {evals} / {max_evals} | Custo: R$ {best_fit:>11,.0f}", end="")
    print()
    return best_dep, best_flt, history

# --------------------------------------------------------
# GA: Algoritmo Genético para otimização da rede logística (Fase 1)
# --------------------------------------------------------

def genetic_algorithm(data, evaluator, max_time=200, max_evals=5000, pop_size=40, cx_rate=0.8, mut_rate=0.3, seed=42, use_ai_start=True, **kwargs):
    random.seed(seed); np.random.seed(seed)
    start_t = time.time(); evals = 0; pop =[]

    P_MAX = 10
    max_open_allowed = min(P_MAX, data.num_candidatos)

    # --- INICIALIZAÇÃO ---
    if use_ai_start:
        # Pool de Sementes de Elite: Baseline (Real ou Proxy) + Arquétipos da IA
        seeds_to_evaluate = []
        seeds_to_evaluate.append(data.get_baseline_depots())
        seeds_to_evaluate.extend(generate_ai_structural_start(data))
        
        for dep_seed in seeds_to_evaluate:
            # Proteção P_MAX
            ativos = [i for i, x in enumerate(dep_seed) if x > 0]
            if len(ativos) > max_open_allowed:
                for i in random.sample(ativos, len(ativos) - max_open_allowed):
                    dep_seed[i] = 0
            if sum(dep_seed) == 0: dep_seed[0] = 2
            
            flt_seed = [2, 3, 2] # Frota coerente de largada
            fit = evaluator.evaluate(dep_seed, flt_seed); evals += 1
            pop.append({'dep': list(dep_seed), 'flt': flt_seed, 'fit': fit})
            
    else:
        # Modo Fatorial (Blind): Garante a criação de pelo menos 1 indivíduo cego para inicializar
        dep_blind = [0] * data.num_candidatos
        min_open = min(2, max_open_allowed)
        num_open = random.randint(min_open, max_open_allowed)
        for idx in random.sample(range(data.num_candidatos), num_open):
            dep_blind[idx] = random.choice([1, 2, 3])
        if sum(dep_blind) == 0: dep_blind[0] = 2
        flt_blind = [random.randint(1, 5) for _ in range(3)]
        
        fit_blind = evaluator.evaluate(dep_blind, flt_blind); evals += 1
        pop.append({'dep': dep_blind, 'flt': flt_blind, 'fit': fit_blind})

    # Completa o resto da população
    while len(pop) < pop_size:
        dep = [0] * data.num_candidatos
        min_open = min(2, max_open_allowed)
        num_open = random.randint(min_open, max_open_allowed)
        for idx in random.sample(range(data.num_candidatos), num_open):
            dep[idx] = random.choice([1, 2, 3])
        if sum(dep) == 0: dep[0] = 2
        flt = [random.randint(1, 5) for _ in range(3)]
        
        fit = evaluator.evaluate(dep, flt); evals += 1
        pop.append({'dep': dep, 'flt': flt, 'fit': fit})

    best = min(pop, key=lambda x: x['fit'])
    history = [(evals, best['fit'])]

    while (time.time() - start_t) < max_time and evals < max_evals:
        new_pop = sorted(pop, key=lambda x: x['fit'])[:2]
        while len(new_pop) < pop_size and evals < max_evals:
            p1 = random.choice(pop[:10]); p2 = random.choice(pop)
            cdep = list(p1['dep']); cflt = list(p1['flt'])
            if random.random() < cx_rate:
                cut = random.randint(0, data.num_candidatos-1)
                cdep[cut:] = p2['dep'][cut:]
                cflt[random.randint(0,2)] = p2['flt'][random.randint(0,2)]
            if random.random() < mut_rate:
                cdep[random.randint(0, data.num_candidatos-1)] = random.choice([0, 1, 2, 3])
                idx = random.randint(0, 2)
                cflt[idx] = max(0, cflt[idx] + random.choice([-1, 1]))
                if sum(cflt)==0: cflt[0]=1

            fit = evaluator.evaluate(cdep, cflt); evals += 1
            new_pop.append({'dep': cdep, 'flt': cflt, 'fit': fit})

        pop = new_pop
        curr = min(pop, key=lambda x: x['fit'])
        if curr['fit'] < best['fit']: best = curr
        history.append((evals, best['fit']))

        elapsed = time.time() - start_t
        if evals % 100 < (pop_size - 2):
            print(f"\r   [{'GA' if use_ai_start else 'GA'}]    Tempo: {elapsed:>5.1f}s / {max_time}s | Evals: {evals} / {max_evals} | Custo: R$ {best['fit']:>11,.0f}", end="")
    print()
    return best['dep'], best['flt'], history

# ==============================================================================
# 4. FASE 2: GERADOR DE COLUNAS A PRIORI (RANDOMIZED CLARKE-WRIGHT)
# ==============================================================================
def generate_route_pool(data, depot_config):
    print(">> Gerando Pool de Rotas (Randomized Clarke-Wright Column Generation)...")

    # --- NOVO: CHECAGEM DE PROXY BASELINE ---
    is_baseline = hasattr(data, 'baseline_config') and depot_config == data.baseline_config
    
    def get_dist(a, b):
        d = data.get_distancia(a, b)
        if is_baseline:
            if a in data.proxy_distances: d += data.proxy_distances[a]
            if b in data.proxy_distances: d += data.proxy_distances[b]
        return d
    # ----------------------------------------
    
    I_active =[i for i, lvl in enumerate(depot_config) if lvl > 0]
    J_all = list(range(data.num_clientes))

    route_pool =[]
    seen_routes = set() # Evita duplicidade de rotas no Gurobi
    r_id = 0

    demanda_ref = np.mean(data.demanda_pico, axis=1)

    for d_idx in I_active:
        for k_idx, v_info in enumerate(data.veiculos_info):
            cap = v_info['cap']

            # 1. Rotas Diretas (Garante viabilidade de visitar todos independentemente)
            for c_idx in J_all:
                seq_tuple = (d_idx, c_idx, d_idx)
                if (k_idx, seq_tuple) not in seen_routes:
                    dist = get_dist(d_idx, c_idx) * 2
                    cost = dist * v_info['var'] + data.custo_visita
                    time_r = (dist / v_info['vel']) + data.tempo_servico
                    route_pool.append({
                        'id': r_id, 'depot': d_idx, 'type': k_idx,
                        'seq': list(seq_tuple), 'clients': [c_idx],
                        'cost': cost, 'time': time_r, 'cap': cap
                    })
                    seen_routes.add((k_idx, seq_tuple))
                    r_id += 1

            # 2. Randomized Clarke-Wright (Roda 10 vezes com ruído para criar diversidade de rotas)
            for iteracao in range(10):
                routes = [[c] for c in J_all]
                route_demands = {i: demanda_ref[c] for i, c in enumerate(J_all)}
                node_to_route = {c: i for i, c in enumerate(J_all)}

                savings =[]
                for i in range(len(J_all)):
                    for j in range(i+1, len(J_all)):
                        c1, c2 = J_all[i], J_all[j]
                        # NOVO: Injeção de Ruído Estocástico (Gama entre 0.8 e 1.2)
                        gamma = random.uniform(0.8, 1.2)
                        sav = get_dist(d_idx, c1) + get_dist(d_idx, c2) - (gamma * get_dist(c1, c2))
                        if sav > 0: savings.append((sav, c1, c2))

                savings.sort(reverse=True, key=lambda x: x[0])

                for sav, c1, c2 in savings:
                    r1 = node_to_route[c1]; r2 = node_to_route[c2]
                    if r1 != r2:
                        if route_demands[r1] + route_demands[r2] <= cap:
                            routes[r1].extend(routes[r2])
                            route_demands[r1] += route_demands[r2]
                            for c in routes[r2]: node_to_route[c] = r1
                            routes[r2] =[]

                # =============================================================
                # REFINAMENTO INTRA-ROTA (BOUNDED 2-OPT) PÓS CLARKE-WRIGHT
                # =============================================================
                for r_seq in routes:
                    if len(r_seq) > 1: # Só faz sentido otimizar se tiver mais de 1 cliente
                        
                        # Monta a sequência completa [Depósito -> Clientes -> Depósito]
                        full_seq = [d_idx] + r_seq + [d_idx]
                        
                        # --- 2-OPT RÁPIDO ---
                        improved = True
                        iters = 0
                        # Limite de 5 iterações é suficiente para polir uma rota pequena
                        while improved and iters < 5: 
                            improved = False
                            # Percorre os nós (ignora os depósitos nas pontas [0] e[len-1] para inverter)
                            for i in range(1, len(full_seq) - 2):
                                for j in range(i + 1, len(full_seq) - 1):
                                    n_prev = full_seq[i-1]
                                    n_i = full_seq[i]
                                    n_j = full_seq[j]
                                    n_next = full_seq[j+1]

                                    d_curr = get_dist(n_prev, n_i) + get_dist(n_j, n_next)
                                    d_new = get_dist(n_prev, n_j) + get_dist(n_i, n_next)

                                    if d_new < d_curr - 1e-4:
                                        full_seq[i:j+1] = full_seq[i:j+1][::-1] # Inverte o trecho
                                        improved = True
                            iters += 1
                        # ---------------------
                        
                        seq_tuple = tuple(full_seq)
                        
                        # Garante que a rota otimizada é inédita no Pool
                        if (k_idx, seq_tuple) not in seen_routes:
                            dist = 0.0
                            for i in range(len(full_seq)-1): 
                                dist += get_dist(full_seq[i], full_seq[i+1])
                            
                            num_clientes = len(full_seq) - 2
                            cost = dist * v_info['var'] + num_clientes * data.custo_visita
                            time_r = (dist / v_info['vel']) + num_clientes * data.tempo_servico
                            
                            route_pool.append({
                                'id': r_id, 'depot': d_idx, 'type': k_idx,
                                'seq': list(seq_tuple), 'clients': full_seq[1:-1],
                                'cost': cost, 'time': time_r, 'cap': cap
                            })
                            seen_routes.add((k_idx, seq_tuple))
                            r_id += 1

    print(f"   Pool Gerado: {len(route_pool)} rotas candidatas únicas.")
    return route_pool

# ==============================================================================
# 5. FASE 2: MILP OPTIMIZER (SET PARTITIONING / PATH-FLOW IRP)
# ==============================================================================
class Phase2MILPResult:
    def __init__(self):
        self.chosen_routes = []
        self.w_dispatch =[]
        self.e2_fleet_usage = []
        self.mass_balance_e1 =[]
        self.fleet_counts = {}
        self.cost_obj = 0
        self.cost_inv = 0

        # --- NOVO: Penalidades Desagregadas ---
        self.cost_pen_bal = 0   # Desvio da Indústria
        self.cost_pen_lost = 0  # Transbordo na Fazenda
        self.cost_pen_over = 0  # Gargalo no Armazém
        self.cost_pen_spot = 0  # Uso de Frota Spot E2

        # --- NOVO: Vamos guardar as quantidades físicas que falharam ---
        self.phys_bal = 0.0   # kg de desvio da indústria
        self.phys_lost = 0.0  # kg de transbordo nas fazendas
        self.phys_over = 0.0  # kg de excesso no armazém
        self.phys_spot = 0    # Quantidade de carretas spot usadas

def solve_phase2_milp(data, depot_config, time_limit=120):
    print("\n>> Executando MILP Fase 2 (Gurobi WLS) - Set Partitioning...")

    I_active =[i for i, lvl in enumerate(depot_config) if lvl > 0]
    J_all = list(range(data.num_clientes))
    K_all = list(range(3))
    T = list(range(data.num_periodos))

    if not I_active or not HAS_GUROBI: return None

    route_pool = generate_route_pool(data, depot_config)
    R_all = [r['id'] for r in route_pool]
    R_k = {k:[r['id'] for r in route_pool if r['type'] == k] for k in K_all}
    R_j = {j: [r['id'] for r in route_pool if j in r['clients']] for j in J_all}

    model = pyo.ConcreteModel()

    # Variáveis E1
    model.Z = pyo.Var(K_all, within=pyo.NonNegativeIntegers)
    model.Y = pyo.Var(R_all, T, within=pyo.NonNegativeIntegers)
    model.Q = pyo.Var(R_all, J_all, T, within=pyo.NonNegativeReals)

    model.InvC = pyo.Var(J_all, T, within=pyo.NonNegativeReals)
    model.LostC = pyo.Var(J_all, T, within=pyo.NonNegativeReals)
    model.ProcessOverflow = pyo.Var(I_active, T, within=pyo.NonNegativeReals)

    # Variáveis E2
    model.InvD = pyo.Var(I_active, T, within=pyo.NonNegativeReals)
    model.W = pyo.Var(I_active, T, within=pyo.NonNegativeIntegers)
    model.Out = pyo.Var(T, within=pyo.NonNegativeReals)
    model.DevPos = pyo.Var(T, within=pyo.NonNegativeReals)
    model.DevNeg = pyo.Var(T, within=pyo.NonNegativeReals)
    model.ExtraW = pyo.Var(T, within=pyo.NonNegativeIntegers)

    # Restrições de Rota

    # 1. Se a rota r for usada no mês t (Y[r,t] > 0), a quantidade total transportada por essa rota não pode exceder a capacidade do veículo
    def route_cap_rule(m, r, t):
        r_info = route_pool[r]
        return sum(m.Q[r, j, t] for j in r_info['clients']) <= r_info['cap'] * m.Y[r, t]
    model.route_cap_c = pyo.Constraint(R_all, T, rule=route_cap_rule)

    # 2. Se a rota r visitar o cliente j no mês t, a quantidade transportada para esse cliente deve ser menor ou igual à demanda máxima do cliente multiplicada por um indicador binário de visita (Y[r,t])
    def route_visit_rule(m, r, j, t):
        r_info = route_pool[r]
        if j in r_info['clients']:
            return m.Q[r, j, t] <= data.cap_estoque_cliente[j] * m.Y[r, t]
        else:
            return m.Q[r, j, t] == 0
    model.route_visit_c = pyo.Constraint(R_all, J_all, T, rule=route_visit_rule)

    # 3. Limite de Frota: O número total de rotas do tipo k usadas no mês t não pode exceder o número de veículos do tipo k multiplicado por um fator de utilização (data.max_rotas)
    def fleet_limit_rule(m, k, t):
        return sum(m.Y[r, t] for r in R_k[k]) <= m.Z[k] * data.max_rotas
    model.fleet_limit_c = pyo.Constraint(K_all, T, rule=fleet_limit_rule)

    # Balanço Fazendas (E1)

    # Para cada cliente j e mês t, o estoque final (InvC) é igual ao estoque inicial (InvC do mês anterior) + demanda do mês - quantidade recebida pelas rotas - quantidade perdida (LostC)
    def inv_cli_rule(m, j, t):
        prev = m.InvC[j, t-1] if t > 0 else 0.0
        return m.InvC[j, t] == prev + data.demanda_pico[j, t] - sum(m.Q[r, j, t] for r in R_j[j]) - m.LostC[j, t]
    model.inv_cli_c = pyo.Constraint(J_all, T, rule=inv_cli_rule)

    # O estoque final em cada cliente não pode exceder a capacidade de estoque do cliente
    def cap_cli_rule(m, j, t): return m.InvC[j, t] <= data.cap_estoque_cliente[j]
    model.cap_cli_c = pyo.Constraint(J_all, T, rule=cap_cli_rule)

    # Balanço Armazéns (E2)

    # Para cada armazém i e mês t, o estoque final (InvD) é igual ao estoque inicial (InvD do mês anterior) + quantidade recebida pelas rotas - quantidade enviada para a indústria (W)
    def inv_dep_rule(m, i, t):
        prev = m.InvD[i, t-1] if t > 0 else 0.0
        R_dep = [r['id'] for r in route_pool if r['depot'] == i]
        inbound = sum(m.Q[r, j, t] for r in R_dep for j in route_pool[r]['clients'])
        return m.InvD[i, t] == prev + inbound - (m.W[i, t] * data.cap_carreta_industria)
    model.inv_dep_c = pyo.Constraint(I_active, T, rule=inv_dep_rule)

    # O estoque final em cada armazém não pode exceder a capacidade do armazém
    def cap_dep_rule(m, i, t): return m.InvD[i, t] <= data.niveis_cap_info[depot_config[i]]['cap']
    model.cap_dep_c = pyo.Constraint(I_active, T, rule=cap_dep_rule)

    # O total recebido por cada armazém i no mês t (inbound) menos o que é enviado para a indústria (W) não pode exceder 150% da capacidade do armazém (ProcessOverflow)
    def max_inbound_rule(m, i, t):
        cap_val = data.niveis_cap_info[depot_config[i]]['cap']
        R_dep = [r['id'] for r in route_pool if r['depot'] == i]
        inbound = sum(m.Q[r, j, t] for r in R_dep for j in route_pool[r]['clients'])
        return inbound - m.ProcessOverflow[i, t] <= 1.5 * cap_val
    model.max_inbound_c = pyo.Constraint(I_active, T, rule=max_inbound_rule)

    # Balanceamento Indústria
    
    # O total enviado para a indústria por cada armazém i no mês t (W) mais o desvio positivo (DevPos) menos o desvio negativo (DevNeg) deve ser igual ao fluxo mensal ideal (data.fluxo_mensal_ideal)
    def max_w_rule(m, t):
        return sum(m.W[i, t] for i in I_active) - m.ExtraW[t] <= data.max_carretas_industria_mes
    model.max_w_c = pyo.Constraint(T, rule=max_w_rule)

    # O total enviado para a indústria por todos os armazéns no mês t (soma de W) deve ser igual ao fluxo mensal ideal (data.fluxo_mensal_ideal) mais o desvio positivo (DevPos) menos o desvio negativo (DevNeg)
    def out_rule(m, t): return m.Out[t] == sum(m.W[i, t] * data.cap_carreta_industria for i in I_active)
    model.out_c = pyo.Constraint(T, rule=out_rule)

    # O total enviado para a indústria por todos os armazéns no mês t (soma de W) mais o desvio positivo (DevPos) menos o desvio negativo (DevNeg) deve ser igual ao fluxo mensal ideal (data.fluxo_mensal_ideal)
    def balance_rule(m, t): return m.Out[t] - data.fluxo_mensal_ideal == m.DevPos[t] - m.DevNeg[t]
    model.bal_c = pyo.Constraint(T, rule=balance_rule)

    # O estoque final no último mês em cada armazém i não pode exceder a capacidade de escoamento da indústria (data.cap_carreta_industria)
    def end_year_rule(m, i): return m.InvD[i, T[-1]] <= data.cap_carreta_industria
    model.end_c = pyo.Constraint(I_active, rule=end_year_rule)

    # Função Objetivo
    # O custo total é composto por:
    # 1. Custo Fixo de Frota: soma do custo fixo anual de cada tipo de veículo multiplicado pelo número de veículos do tipo k (Z[k]) e pelo Fator NPV (Net Present Value)
    # 2. Custo Variável de Rotas: soma do custo de cada rota multiplicado pelo número de vezes que a rota é usada (Y[r,t])
    # 3. Custo de Estoque: soma do custo de manter estoque nos clientes (InvC) e nos armazéns (InvD) multiplicado pelos respectivos custos de estoque
    # 4. Penalidades: soma das penalidades por desvio de equilíbrio (DevPos, DevNeg), transbordo (LostC), excesso de inbound (ProcessOverflow) e uso de frota spot (ExtraW) multiplicado pelos respectivos grandes M
    def obj_rule(m):
        # Frota: Custo Anual * Fator NPV
        fleet_cost = sum(data.veiculos_info[k]['fixo'] * 12 * data.factor_fleet * m.Z[k] for k in K_all)
        route_cost = sum(route_pool[r]['cost'] * m.Y[r, t] for r in R_all for t in T)

        hold_c = sum(m.InvC[j,t] * data.custo_estoque_cli for j in J_all for t in T)
        hold_d = sum(m.InvD[i,t] * data.custo_estoque_dep for i in I_active for t in T)

        pen_bal = sum((m.DevPos[t] + m.DevNeg[t]) * data.big_m_dev for t in T)
        pen_lost = sum(m.LostC[j, t] * data.big_m_lost for j in J_all for t in T)
        pen_over = sum(m.ProcessOverflow[i, t] * data.big_m_over for i in I_active for t in T)
        pen_extra_w = sum(m.ExtraW[t] * data.big_m_spot for t in T)

        # Agrupamento das Penalidades Big-M do Gurobi
        total_penalties = pen_bal + pen_lost + pen_over + pen_extra_w

        # Aplica o LCC nas Rotas, Estoques e nas Penalidades simultaneamente!
        return fleet_cost + (route_cost + hold_c + hold_d + total_penalties) * data.factor_lcc

    model.obj = pyo.Objective(rule=obj_rule, sense=pyo.minimize)
    
    # Configurações do Solver
    solver = pyo.SolverFactory('gurobi_direct')

    solver.options['TimeLimit'] = time_limit
    solver.options['MIPFocus'] = 1
    solver.options['MIPGap'] = 0.01

    # --- NOVAS OPÇÕES PARA PREVENIR OUT-OF-MEMORY (OOM) ---
    # Descarrega a árvore de busca no SSD quando atingir 24.0 GB de RAM
    solver.options['NodefileStart'] = 24.0
    solver.options['NodefileDir'] = 'F:/gurobi_nodes' 
    # Limita as threads (cada thread extra consome mais memória ram). 
    solver.options['Threads'] = 12
    # ------------------------------------------------------

    try:
        results = solver.solve(model, tee=True)
        res = Phase2MILPResult()
        res.cost_obj = pyo.value(model.obj)
        res.fleet_counts ={k: int(pyo.value(model.Z[k])) for k in K_all}

        # Extração dos Custos de Estoque
        hold_c = sum(pyo.value(model.InvC[j,t]) * data.custo_estoque_cli for j in J_all for t in T)
        hold_d = sum(pyo.value(model.InvD[i,t]) * data.custo_estoque_dep for i in I_active for t in T)
        res.cost_inv = hold_c + hold_d

        # --- CORREÇÃO: Extrai as Quantidades Físicas (Sem o Big-M) ---
        res.phys_bal = sum(pyo.value(model.DevPos[t]) + pyo.value(model.DevNeg[t]) for t in T)
        res.phys_lost = sum(pyo.value(model.LostC[j, t]) for j in J_all for t in T)
        res.phys_over = sum(pyo.value(model.ProcessOverflow[i, t]) for i in I_active for t in T)
        res.phys_spot = sum(pyo.value(model.ExtraW[t]) for t in T)
        # -------------------------------------------------------------

        for t in T:
            mes_nome = data.nomes_periodos[t]

            total_w_mes = sum(pyo.value(model.W[i, t]) for i in I_active)
            extra_w_val = pyo.value(model.ExtraW[t])
            res.e2_fleet_usage.append({
                'Mês': mes_nome,
                'Carretas Normais Utilizadas': total_w_mes - extra_w_val,
                'Limite Disponível Indústria': data.max_carretas_industria_mes,
                'Carretas SPOT (Emergenciais)': extra_w_val,
                'Volume Total Escoado (kg)': total_w_mes * data.cap_carreta_industria
            })

            for j in J_all:
                inv_inicial = pyo.value(model.InvC[j, t-1]) if t > 0 else 0.0
                gerado = data.demanda_pico[j, t]
                coletado = sum(pyo.value(model.Q[r, j, t]) for r in R_j[j])
                inv_final = pyo.value(model.InvC[j, t])
                lost_val = pyo.value(model.LostC[j, t])
                cap_max = data.cap_estoque_cliente[j]

                res.mass_balance_e1.append({
                    'Cliente': data.df_cluster.iloc[j]['Município'], 'Mês': mes_nome, 'Cap. Max': round(cap_max, 1),
                    'Est. Inicial': round(inv_inicial, 1), 'Gerado': round(gerado, 1), 'Disponível': round(inv_inicial + gerado, 1),
                    'Coletado': round(coletado, 1), 'Transbordo': round(lost_val, 1), 'Est. Final': round(inv_final, 1),
                    '% Ocupação': round((inv_final / cap_max) * 100, 1) if cap_max > 0 else 0
                })

            for r in R_all:
                y_val = pyo.value(model.Y[r, t])
                if y_val > 0.5:
                    y_int = int(round(y_val)) # Número exato de vezes que a rota foi usada
                    
                    for _ in range(y_int):
                        # CORREÇÃO: Divide o volume total (Q) pelo número de viagens (y_int)
                        col_dict = {
                            j: (pyo.value(model.Q[r, j, t]) / y_int) 
                            for j in route_pool[r]['clients'] 
                            if pyo.value(model.Q[r, j, t]) > 0.1
                        }
                        
                        if col_dict:
                            res.chosen_routes.append({
                                'Mês': mes_nome, 
                                'route_obj': route_pool[r], 
                                'collections': col_dict
                            })

            for i in I_active:
                w_val = pyo.value(model.W[i, t])
                if w_val > 0.5: res.w_dispatch.append({'Mês': mes_nome, 'Depot': i, 'Carretas': int(w_val)})
            
        # --- LIMPEZA DE MEMÓRIA DO PYOMO/GUROBI ---
        del model
        del solver
        gc.collect()
        # ------------------------------------------

        return res
    except Exception as e:
        print(f"[!] Erro no MILP: {e}")
        return None

# ==============================================================================
# 6. FASE 3: MILP DE ALOCAÇÃO OPERACIONAL E SEQUENCIADOR
# ==============================================================================
class FinalSolution:
    def __init__(self):
        self.routes =[]; self.fleet_utilization =[]; self.facility_utilization =[]
        self.mass_balance_e1 =[]; self.mass_balance_e2 =[]; self.e2_fleet_usage =[]
        self.cost_breakdown ={}; self.total_cost = 0; self.open_depots =[]; self.fleet_counts = {}

def build_final_solution(data, depot_config, milp_res, time_limit=120):
    
    # --- NOVO: CHECAGEM DE PROXY BASELINE ---
    is_baseline = hasattr(data, 'baseline_config') and depot_config == data.baseline_config
    
    def get_dist(a, b):
        d = data.get_distancia(a, b)
        if is_baseline:
            if a in data.proxy_distances: d += data.proxy_distances[a]
            if b in data.proxy_distances: d += data.proxy_distances[b]
        return d
    # ----------------------------------------

    sol = FinalSolution()
    fleet_config = milp_res.fleet_counts
    sol.fleet_counts = {data.veiculos_info[k]['id']: fleet_config[k] for k in range(3)}
    sol.mass_balance_e1 = milp_res.mass_balance_e1
    sol.e2_fleet_usage = milp_res.e2_fleet_usage

    I_active =[i for i, lvl in enumerate(depot_config) if lvl > 0]
    for i in I_active:
        lvl = depot_config[i]
        sol.open_depots.append({'ID': f'Dep_{i}', 'Município': data.df_cluster.iloc[i]['Município'],
                                'Nível': data.niveis_cap_info[lvl]['nome'], 'Cap': data.niveis_cap_info[lvl]['cap']})

    capex = sum(data.niveis_cap_info[depot_config[i]]['capex'] for i in I_active)
    # OPEX e Frota usam o novo fator NPV
    opex = sum(data.niveis_cap_info[depot_config[i]]['opex'] * 12 * data.factor_opex for i in I_active)
    fleet_fix = sum(data.veiculos_info[k]['fixo'] * 12 * data.factor_fleet * fleet_config[k] for k in range(3))

    var_transp = var_time = var_stop = 0

    for t_idx, mes in enumerate(data.nomes_periodos):
        active_fleet =[]
        for k, qtd in fleet_config.items():
            for n in range(qtd):
                active_fleet.append({'id': f"{data.veiculos_info[k]['id']}_{n+1}", 'type': k, 'pos': None, 'time': 0, 'load': 0})

        depot_loads = {i: 0.0 for i in I_active}
        routes_t =[cr for cr in milp_res.chosen_routes if cr['Mês'] == mes]

        # --- FASE 3: MILP ASSIGNMENT (Alocação Exata de Frota) ---
        for k in range(3):
            comp_routes =[r for r in routes_t if r['route_obj']['type'] == k]
            comp_vehicles =[v for v in active_fleet if v['type'] == k]

            if not comp_routes: continue
            if not comp_vehicles: continue

            m = pyo.ConcreteModel()
            m.R = pyo.RangeSet(0, len(comp_routes)-1)
            m.V = pyo.RangeSet(0, len(comp_vehicles)-1)
            unique_depots = list(set(r['route_obj']['depot'] for r in comp_routes))
            m.D = pyo.Set(initialize=unique_depots)

            m.X = pyo.Var(m.R, m.V, within=pyo.Binary)
            m.U = pyo.Var(m.D, m.V, within=pyo.Binary)
            m.OT = pyo.Var(m.V, within=pyo.NonNegativeReals)
            m.Wmax = pyo.Var(within=pyo.NonNegativeReals)

            # Cada rota é designada exatamente para 1 veículo
            def c1_rule(m, r): return sum(m.X[r, v] for v in m.V) == 1
            m.c1 = pyo.Constraint(m.R, rule=c1_rule)

            # Se um veículo v é designado para uma rota que parte do depósito d, então o veículo deve ser marcado como ativo para esse depósito
            def c2_rule(m, r, v):
                d = comp_routes[r]['route_obj']['depot']
                return m.X[r, v] <= m.U[d, v]
            m.c2 = pyo.Constraint(m.R, m.V, rule=c2_rule)

            # Dados do tipo de veículo para calcular tempos e custos de deadhead
            v_info = data.veiculos_info[k]

            # Funções auxiliares para calcular tempos e custos de deadhead com base na posição atual do veículo e no depósito da rota
            def get_dh_time(v_idx, d):
                pos = comp_vehicles[v_idx]['pos']
                if pos is None or pos == d: return 0.0
                return get_dist(pos, d) / v_info['vel']

            # Custo de deadhead é proporcional à distância percorrida sem carga, multiplicado pelo custo variável do veículo
            def get_dh_cost(v_idx, d):
                pos = comp_vehicles[v_idx]['pos']
                if pos is None or pos == d: return 0.0
                return get_dist(pos, d) * v_info['var']

            # Tempo total da rota r, incluindo o tempo de serviço em cada cliente e o tempo de deslocamento entre os clientes, calculado com base na sequência ativa da rota (considerando apenas os clientes que realmente serão coletados)
            def get_r_time(r):
                r_data = comp_routes[r]
                dep_idx = r_data['route_obj']['depot']
                col_dict = r_data['collections']
                seq_ativa = [dep_idx] + [c for c in r_data['route_obj']['seq'][1:-1] if c in col_dict] + [dep_idx]
                r_dist = sum(get_dist(seq_ativa[i], seq_ativa[i+1]) for i in range(len(seq_ativa)-1))
                return (r_dist / v_info['vel']) + len(col_dict) * data.tempo_servico

            # Pré-calcula os tempos das rotas para usar nas restrições de tempo total
            r_times = {r: get_r_time(r) for r in m.R}

            # Tempo total trabalhado por veículo v deve ser menor ou igual à jornada de trabalho, considerando o tempo da rota e o tempo de deadhead para chegar ao depósito da rota
            def ot_rule(m, v):
                wv = sum(m.X[r, v] * r_times[r] for r in m.R) + sum(m.U[d, v] * get_dh_time(v, d) for d in m.D)
                return m.OT[v] >= wv - data.horas_trabalho
            m.ot_c = pyo.Constraint(m.V, rule=ot_rule)

            # O tempo máximo trabalhado por qualquer veículo (Wmax) deve ser maior ou igual ao tempo total trabalhado por cada veículo v, para que possamos minimizar Wmax e balancear a carga de trabalho entre os veículos
            def wmax_rule(m, v):
                wv = sum(m.X[r, v] * r_times[r] for r in m.R) + sum(m.U[d, v] * get_dh_time(v, d) for d in m.D)
                return m.Wmax >= wv
            m.wmax_c = pyo.Constraint(m.V, rule=wmax_rule)

            # Função Objetivo: Minimizar o custo total, que é composto pelo custo de horas extras (OT) multiplicado pelo custo de hora extra do veículo, mais o custo de deadhead (U) multiplicado pelo custo de deadhead, mais um pequeno termo para minimizar Wmax e incentivar uma distribuição equilibrada das rotas entre os veículos
            def obj_rule(m):
                ot_cost = sum(m.OT[v] * ((data.gamma_hora_extra - 1) * v_info['hora']) for v in m.V)
                dh_cost = sum(m.U[d, v] * get_dh_cost(v, d) for d in m.D for v in m.V)
                return ot_cost + dh_cost + 0.01 * m.Wmax
            m.obj = pyo.Objective(rule=obj_rule, sense=pyo.minimize)

            solver = pyo.SolverFactory('gurobi_direct')
            #solver.options['OutputFlag'] = 0
            solver.options['TimeLimit'] = time_limit
            solver.options['MIPFocus'] = 1
            solver.options['MIPGap'] = 0.01

            # --- PROTEÇÃO OOM PARA A FASE 3 ---
            solver.options['Threads'] = 12  # Fase 3 é rápida, não precisa de muitas threads
            # ----------------------------------

            # solver.solve(m, tee=True)
            solver.solve(m)

            # Aplica a alocação exata do MILP aos veículos
            for v_idx in m.V:
                assigned_r_indices =[r for r in m.R if pyo.value(m.X[r, v_idx]) > 0.5]
                if not assigned_r_indices: continue

                v_obj = comp_vehicles[v_idx]
                assigned_r_indices.sort(key=lambda r: comp_routes[r]['route_obj']['depot'])

                for r in assigned_r_indices:
                    r_data = comp_routes[r]
                    dep_idx = r_data['route_obj']['depot']
                    col_dict = r_data['collections']
                    seq_ativa = [dep_idx] +[c for c in r_data['route_obj']['seq'][1:-1] if c in col_dict] + [dep_idx]

                    r_dist = sum(get_dist(seq_ativa[i], seq_ativa[i+1]) for i in range(len(seq_ativa)-1))
                    r_time = (r_dist / v_info['vel']) + len(col_dict) * data.tempo_servico
                    r_load = sum(col_dict.values())

                    dead_d = dead_t = 0
                    if v_obj['pos'] is not None and v_obj['pos'] != dep_idx:
                        dead_d = get_dist(v_obj['pos'], dep_idx)
                        dead_t = dead_d / v_info['vel']
                        var_transp += dead_d * v_info['var']
                        var_time += dead_t * v_info['hora']
                        v_obj['time'] += dead_t

                    v_obj['pos'] = dep_idx
                    v_obj['time'] += r_time
                    v_obj['load'] += r_load
                    var_transp += r_dist * v_info['var']
                    var_time += r_time * v_info['hora']
                    var_stop += len(col_dict) * data.custo_visita
                    depot_loads[dep_idx] += r_load

                    sol.routes.append({
                        'Período': mes, 'Veículo ID': v_obj['id'], 'Tipo': v_info['id'],
                        'Origem': data.df_cluster.iloc[dep_idx]['Município'],
                        'Sequência': " -> ".join([str(x) for x in seq_ativa]),
                        'Dist (km)': round(r_dist, 1), 'Carga (kg)': round(r_load, 1), 'Deadhead': round(dead_d, 1)
                    })

            # --- LIMPEZA DE MEMÓRIA DENTRO DO LOOP DA FASE 3 ---
            del m
            del solver
            gc.collect()
            # ---------------------------------------------------

        for v in active_fleet:
            if v['time'] > data.horas_trabalho:
                var_time += (data.gamma_hora_extra - 1) * data.veiculos_info[v['type']]['hora'] * (v['time'] - data.horas_trabalho)
            sol.fleet_utilization.append({'Mês': mes, 'Veículo': v['id'], 'Horas': round(v['time'],1), 'Carga Total': round(v['load'],1)})

        for i in I_active:
            cap = data.niveis_cap_info[depot_config[i]]['cap']
            pct = (depot_loads[i] / cap)*100 if cap>0 else 0
            sol.facility_utilization.append({'Mês': mes, 'Depósito': data.df_cluster.iloc[i]['Município'], 'Processado': round(depot_loads[i],1), '% Ocupação': round(pct,1)})

        for d in[w for w in milp_res.w_dispatch if w['Mês'] == mes]:
            sol.mass_balance_e2.append({'Mês': mes, 'Depósito': data.df_cluster.iloc[d['Depot']]['Município'], 'Carretas Despachadas': d['Carretas'], 'Volume (kg)': d['Carretas']*data.cap_carreta_industria})

    #=========================================================================
    # APLICAÇÃO DOS CUSTOS REAIS DE NÃO-CONFORMIDADE (SHADOW PRICING)
    # Convertendo a "Penalidade Matemática" em "Fatura Financeira Real"
    # =========================================================================
    real_cost_bal  = milp_res.phys_bal * data.custo_real_dev
    real_cost_lost = milp_res.phys_lost * data.custo_real_lost
    real_cost_over = milp_res.phys_over * data.custo_real_over
    real_cost_spot = milp_res.phys_spot * data.custo_real_spot
    
    # Para o LCC (15 anos), multiplicamos apenas os danos operacionais recorrentes por Lambda
    sol.cost_breakdown = {
        'CAPEX': capex, 
        'OPEX': opex, 
        'Fleet': fleet_fix,
        'Logistics (E1)': var_transp * data.factor_lcc, 
        'Time': var_time * data.factor_lcc,
        'Stops': var_stop * data.factor_lcc,
        'Inventory': milp_res.cost_inv * data.factor_lcc,

        # Multas reais multiplicadas pelo Fator LCC Variável
        'Penalty: Flow (E2)': real_cost_bal * data.factor_lcc,
        'Penalty: Retailer': real_cost_lost * data.factor_lcc,
        'Penalty: Facility': real_cost_over * data.factor_lcc,
        'Penalty: Fleet (E2)': real_cost_spot * data.factor_lcc
    }

    # O Total agora é um Custo Financeiro Real e Factível!
    sol.total_cost = sum(sol.cost_breakdown.values())
    return sol

# ==============================================================================
# 7. FUNÇÕES DE PLOTAGEM E SALVAMENTO
# ==============================================================================

# ==============================================================================
# 7.1 Mapas das Rotas (Mapas Poligonais com Clusters e Depósitos)
# ==============================================================================

def gerar_mapa_poligonal(sol, data, method_name):

    map_files = []
    cmap = plt.get_cmap('tab20')
    depot_colors = {d['ID']: mcolors.to_hex(cmap(i % 20)) for i, d in enumerate(sol.open_depots)}

    # 1. Identificar os municípios ativos nesta instância específica
    # CORREÇÃO: Pega o nome raiz (remove " (Polo A)", etc) para casar com o GeoJSON
    active_muns = data.df_cluster['Município_Norm'].apply(lambda x: x.split('(')[0].strip()).unique()

    # 2. Verificar se a instância é um subconjunto (menor que o estado inteiro)
    is_sub_instance = len(active_muns) < len(data.gdf_estado)

    for t_idx, mes in enumerate(data.nomes_periodos):
        fig, ax = plt.subplots(figsize=(10, 12))

        routes_t = [r for r in sol.routes if r['Período'] == mes]

        # 3. Mapear os NOMES RAÍZES para os depósitos que os atendem
        mun_serving_depots = {}
        for r in routes_t:
            seq = [int(x) for x in r['Sequência'].split(' -> ')]
            dep_id = f"Dep_{seq[0]}"
            for cli in seq[1:-1]:
                # Busca o nome padronizado do dataframe
                mun_name_raw = data.df_cluster.iloc[cli]['Município_Norm']
                # Limpa o sufixo artificial das instâncias upscaled
                mun_name_base = mun_name_raw.split('(')[0].strip()
                
                if mun_name_base not in mun_serving_depots:
                    mun_serving_depots[mun_name_base] = set()
                mun_serving_depots[mun_name_base].add(dep_id)

        data.gdf_estado['color'] = '#f0f0f0' # Cor de fundo padrão
        data.gdf_estado['hatch'] = None

        # Preencher as cores do GeoDataFrame baseado no atendimento
        for idx, row in data.gdf_estado.iterrows():
            m_name = row.get('name_upper')
            if pd.notna(m_name) and m_name in mun_serving_depots:
                servs = list(mun_serving_depots[m_name])
                if len(servs) == 1:
                    # Atendido exclusivamente por um armazém
                    data.gdf_estado.at[idx, 'color'] = depot_colors[servs[0]]
                else:
                    # Split Delivery (Nó A por um armazém, Nó B por outro, ou múltiplas frotas)
                    data.gdf_estado.at[idx, 'color'] = '#cccccc'
                    data.gdf_estado.at[idx, 'hatch'] = '///'

        # PLOTAGEM 1: Polígonos Base
        data.gdf_estado.plot(ax=ax, color=data.gdf_estado['color'], edgecolor='white', lw=0.5)

        # PLOTAGEM 2: Hachuras para Split Delivery
        hatch_geo = data.gdf_estado[data.gdf_estado['hatch'].notna()]
        if not hatch_geo.empty:
            hatch_geo.plot(ax=ax, facecolor='none', hatch='///', edgecolor='#333', lw=0.5)

        # PLOTAGEM 3: PERÍMETRO DA INSTÂNCIA (Apenas se for um recorte do estado)
        gdf_instance = data.gdf_estado[data.gdf_estado['name_upper'].isin(active_muns)]
        if is_sub_instance and not gdf_instance.empty:
            instance_boundary = gdf_instance.dissolve()
            instance_boundary.boundary.plot(ax=ax, edgecolor='#1a1a1a', linewidth=2.5, zorder=3)

        # Centralizar o zoom apenas na área ativa
        if not gdf_instance.empty:
            b = gdf_instance.total_bounds
            ax.set_xlim(b[0]-0.2, b[2]+0.2)
            ax.set_ylim(b[1]-0.2, b[3]+0.2)

        # PLOTAGEM DAS ROTAS
        ls_map = {'VUC': '-', 'Truck': '--', 'Carreta': ':'}
        for r in routes_t:
            seq = [int(x) for x in r['Sequência'].split(' -> ')]
            xs = [data.df_cluster.iloc[n]['Longitude'] for n in seq]
            ys = [data.df_cluster.iloc[n]['Latitude'] for n in seq]
            ax.plot(xs, ys, color='#222', ls=ls_map.get(r['Tipo'], '-'), lw=1.2, alpha=0.7, zorder=4)

        # PLOTAGEM DOS ARMAZÉNS
        patches = []
        for d in sol.open_depots:
            did = d['ID']
            mid = int(did.split('_')[1])
            px = data.df_cluster.iloc[mid]['Longitude']
            py = data.df_cluster.iloc[mid]['Latitude']
            c = depot_colors[did]

            nivel = d.get('Nível', 'Médio')
            if nivel in ['Pequeno', 'Small']: marker_size = 150
            elif nivel in ['Médio', 'Medium']: marker_size = 300
            elif nivel in ['Grande', 'Large']: marker_size = 500
            else: marker_size = 200

            ax.scatter(px, py, s=marker_size, c=[c], edgecolors='k', zorder=5)
            patches.append(mpatches.Patch(color=c, label=f"Depot {mid} ({nivel})"))

        # PLOTAGEM DOS NÓS ARTIFICIAIS (Apenas para as instâncias maiores que 139)
        if data.num_candidatos > 139:
            sub_xs = []
            sub_ys = []
            for _, row_df in data.df_cluster.iterrows():
                if "(" in row_df['Município']:  # Indica que é um Polo A/B
                    sub_xs.append(row_df['Longitude'])
                    sub_ys.append(row_df['Latitude'])
            
            if sub_xs:
                ax.scatter(sub_xs, sub_ys, color='#1f77b4', s=60, marker='*', edgecolor='black', zorder=6)
                star = mlines.Line2D([], [], color='#1f77b4', marker='*', linestyle='None', 
                                     markersize=8, markeredgecolor='black', label='Artificial Sub-Node')
                patches.append(star)

        patches.append(mpatches.Patch(facecolor='#cccccc', hatch='///', label='Split Delivery', edgecolor='k'))

        ax.legend(handles=patches, loc='upper left', bbox_to_anchor=(1,1), fontsize='small', title="Active Installations")
        ax.set_title(f"Rede 2E-LIRP - {mes} ({method_name})")
        ax.axis('off')

        fname = f"Mapa_{method_name}_{t_idx+1:02d}_{mes}.png"
        plt.savefig(fname, dpi=300, bbox_inches='tight')
        plt.close(fig)
        map_files.append(fname)

    return map_files

# ==============================================================================
# 7.2 Gráficos de Comparação de Custos (Bar Plots com Breakdown Detalhado)
# ==============================================================================

def plot_cost_comparison_v3(results_dict):
    
    df_list =[]
    
    # 1. BASELINE ESTIMADO REAL (COM INTERVALOS DE INEFICIÊNCIA ESPECÍFICOS)
    if 'BASELINE' in results_dict:
        sol_b = results_dict['BASELINE']
        
        # Gaps de ineficiência empíricos da literatura
        gaps = {
            'Routing'  : [0.05, 0.10, 0.15], # Logística, Tempo, Paradas
            'Inventory': [0.10, 0.15, 0.20], # Estoque
            'Penalties': [0.15, 0.225, 0.30] # Todas as Multas
        }
        
        for i in range(3):
            total_est = 0
            for comp, val in sol_b.cost_breakdown.items():
                if comp in['Logistics (E1)', 'Time', 'Stops']:
                    est_val = val * (1.0 + gaps['Routing'][i])
                elif 'Inventory' in comp:
                    est_val = val * (1.0 + gaps['Inventory'][i])
                elif 'Penalty' in comp or 'Fine' in comp:
                    est_val = val * (1.0 + gaps['Penalties'][i])
                else:
                    est_val = val # CAPEX, OPEX e Frota Fixa (Sem ineficiência operacional)
                
                df_list.append({'Method': 'BASELINE (Estimated)', 'Component': comp, 'Cost': est_val})
                total_est += est_val
                
            df_list.append({'Method': 'BASELINE (Estimated)', 'Component': 'TOTAL', 'Cost': total_est})

    # 2. RESULTADOS DOS ALGORITMOS MATEMÁTICOS EXATOS
    name_map = {
        'BASELINE': 'BASELINE (Estimated)',
        'BASELINE': 'BASELINE (Optimized)',
        'SA': 'SA',
        'GA': 'GA'
    }
    
    for method, sol in results_dict.items():
        m_name = name_map.get(method, method)
        for comp, val in sol.cost_breakdown.items():
            df_list.append({'Method': m_name, 'Component': comp, 'Cost': val})
        df_list.append({'Method': m_name, 'Component': 'TOTAL', 'Cost': sol.total_cost})

    df = pd.DataFrame(df_list)
    
    # Remove componentes completamente zerados em todos os métodos para não poluir o gráfico
    df = df.groupby('Component').filter(lambda x: x['Cost'].max() > 100)
    
    # 3. SEPARAÇÃO DOS COMPONENTES NOS 4 SUBGRÁFICOS
    grupo_infra = ['CAPEX', 'OPEX', 'Fleet']
    grupo_operacao =['Logistics (E1)', 'Time', 'Stops', 'Inventory']
    grupo_penalidades = [c for c in df['Component'].unique() if 'Penalty: ' in c or 'Fine' in c]
    
    df_g1 = df[df['Component'].isin(grupo_infra)]
    df_g2 = df[df['Component'].isin(grupo_operacao)]
    df_g3 = df[df['Component'].isin(grupo_penalidades)]
    df_g4 = df[df['Component'] == 'TOTAL']
    
    # 4. CONFIGURAÇÃO VISUAL DO PLOT QUÁDRUPLO
    # As larguras (width_ratios) são proporcionais ao número de colunas em cada gráfico
    fig, axes = plt.subplots(1, 4, figsize=(24, 8), gridspec_kw={'width_ratios':[1.2, 2.5, 2.0, 0.6]})
    
    custom_palette = {
        'BASELINE (Estimated)': '#8B0000',  # Vermelho Escuro
        'BASELINE (Optimized)': '#FF8C00',  # Laranja
        'SA': "#1F77B4",                    # Azul Escuro
        'GA': "#2CA02C"                     # Verde Água
    }
    
    # --- PAINEL 1: Infraestrutura (CAPEX e OPEX) ---
    sns.barplot(data=df_g1, x='Component', y='Cost', hue='Method', 
                palette=custom_palette, errorbar=None, ax=axes[0])
    axes[0].set_title('Fixed costs', fontsize=13, fontweight='bold')
    axes[0].set_xticks(axes[0].get_xticks())
    axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=30, ha='right', fontsize=11)
    # axes[0].yaxis.set_major_formatter(ticker.FuncFormatter(lambda y, _: f'R$ {y:,.0f}'))

    # Formatação limpa do eixo Y (mantendo a notação científica da Matplotlib)
    formatter = ticker.ScalarFormatter(useMathText=True)
    formatter.set_scientific(True)
    formatter.set_powerlimits((-2, 4))
    axes[0].yaxis.set_major_formatter(formatter)

    axes[0].set_ylabel('Cost', fontsize=12)
    axes[0].get_legend().remove()
    
    # --- PAINEL 2: Operação Logística e Frota ---
    if not df_g2.empty:
        sns.barplot(data=df_g2, x='Component', y='Cost', hue='Method', 
                    palette=custom_palette, errorbar=lambda x: (x.min(), x.max()), 
                    capsize=0.1, err_kws={'linewidth': 1.5, 'color': 'black'}, ax=axes[1])
    axes[1].set_title('Variable costs', fontsize=13, fontweight='bold')
    axes[1].set_xticks(axes[1].get_xticks())
    axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=30, ha='right', fontsize=11)
    # axes[1].yaxis.set_major_formatter(ticker.FuncFormatter(lambda y, _: f'R$ {y:,.0f}'))
    axes[1].yaxis.set_major_formatter(formatter)
    axes[1].set_ylabel('')
    axes[1].get_legend().remove()

    # --- PAINEL 3: Penalidades do Sistema ---
    if not df_g3.empty:
        sns.barplot(data=df_g3, x='Component', y='Cost', hue='Method', 
                    palette=custom_palette, errorbar=lambda x: (x.min(), x.max()), 
                    capsize=0.1, err_kws={'linewidth': 1.5, 'color': 'black'}, ax=axes[2])
    axes[2].set_title('Penalties (Soft Constraints)', fontsize=13, fontweight='bold')
    axes[2].set_xticks(axes[2].get_xticks())
    axes[2].set_xticklabels([c.get_text().replace('Penalty: ', '') for c in axes[2].get_xticklabels()], rotation=30, ha='right', fontsize=11)
    # axes[2].yaxis.set_major_formatter(ticker.FuncFormatter(lambda y, _: f'R$ {y:,.0f}'))
    axes[2].yaxis.set_major_formatter(formatter)
    axes[2].set_ylabel('')
    if axes[2].get_legend() is not None: axes[2].get_legend().remove()

    # --- PAINEL 4: Custo Total LCC ---
    sns.barplot(data=df_g4, x='Component', y='Cost', hue='Method', 
                palette=custom_palette, errorbar=lambda x: (x.min(), x.max()), 
                capsize=0.1, err_kws={'linewidth': 1.5, 'color': 'black'}, ax=axes[3])
    axes[3].set_title('Total LCC', fontsize=13, fontweight='bold')
    axes[3].set_xticks(axes[3].get_xticks())
    axes[3].set_xticklabels(['Total'], rotation=30, ha='right', fontsize=12, fontweight='bold')
    # axes[3].yaxis.set_major_formatter(ticker.FuncFormatter(lambda y, _: f'R$ {y:,.0f}'))
    axes[3].yaxis.set_major_formatter(formatter)
    axes[3].set_ylabel('')
    
    # Legenda Global (posicionada fora à direita do último gráfico)
    axes[3].legend(title='Scenarios and Algorithms', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=11, title_fontsize=12)
    
    plt.suptitle('Financial Comparison: Real Operation Estimate vs. 2E-LIRP Optimized', fontsize=16, fontweight='bold', y=1.05)
    plt.tight_layout()
    plt.savefig('Comparativo_Custos_V3.png', dpi=300, bbox_inches='tight')

# ==============================================================================
# 7.3 Gráfico de Evolução da Demanda vs. Capacidade Estática da Rede
# ==============================================================================

def plot_capacity_vs_demand(results_dict, data):
    
    # =========================================================================
    # 1. CÁLCULO DA DEMANDA ANUAL E DADOS HISTÓRICOS (S-CURVE INTEGRATION)
    # =========================================================================
    ultimo_ano_hist = 2024
    anos_futuros = np.arange(ultimo_ano_hist + 1, ultimo_ano_hist + 1 + data.horizonte_anos)
    
    # Demanda anual base (soma de todas as fazendas e meses no ano base atual/2024)
    base_annual_demand = np.sum(data.demanda_base)
    
    # Projeção S-Curve usando os multiplicadores estocásticos calculados no Data object
    demand_by_year = base_annual_demand * np.array(data.curva_crescimento_demanda[:data.horizonte_anos])
    demand_p95 = base_annual_demand * np.array(data.fator_pico_serie[:data.horizonte_anos])
    
    # Reconstrução da trajetória histórica real (conforme plantio_regional_norte_6_0.py)
    anos_hist = np.arange(2000, 2025)
    total_mock = np.array([
        290097, 309529, 354883, 409264, 541537, 687118, 596696, 583585,
        646558, 612324, 664195, 705878, 730965, 834257, 1034697, 1199494,
        1227972, 1278462, 1348373, 1413461, 1539685, 1730038, 1878240, 2144655, 2193201
    ])
    
    # Parâmetros base da versão Plantio Regional Norte 6.0 New para converter o "total_mock" em kg de demanda histórica real
    THETA_BASE_POND = 12.63
    omega_conversion = 0.055971 # Omega (2039) - Fator_Omega_5.0
    massa_historica_calc = total_mock * THETA_BASE_POND * omega_conversion
    
    # Escalonamento para garantir conexão contínua e suave:
    # Ajusta o formato histórico do estado do Tocantins com o seu número base atual para evitar "degraus"
    fator_escala = base_annual_demand / massa_historica_calc[-1]
    massa_historica = massa_historica_calc * fator_escala
    
    # Vetores de conexão para o gráfico plotar sem "buracos" entre 2024 e 2025
    anos_conexao = np.insert(anos_futuros, 0, ultimo_ano_hist)
    media_conexao = np.insert(demand_by_year, 0, base_annual_demand)
    p95_conexao = np.insert(demand_p95, 0, base_annual_demand)
    
    print(f"\n{'='*60}")
    print(f"📶  Demanda Projetada com Multiplicadores Estocásticos (S-Curve):")
    print(f"{'='*60}\n")
    print(f"1️⃣  Ano Base ({ultimo_ano_hist}): {base_annual_demand:,.0f} kg")
    print(f"✅  Ano Horizonte ({anos_futuros[-1]}): {demand_by_year[-1]:,.0f} kg")
    print(f"🚩  Fator de crescimento total esperado: {data.curva_crescimento_demanda[-1]:.4f}x")
    print(f"🚨  Fator de Pico de Estresse (P95): {data.fator_pico_serie[-1]:.4f}x")
    
    fig, ax = plt.subplots(figsize=(14, 8))
    
    custom_palette = {
        'BASELINE': '#FF8C00', # Laranja
        'SA': "#1F77B4",       # Azul Escuro
        'GA': "#2CA02C"        # Verde Água
    }
    
    # =========================================================================
    # 2. PLOTAGEM DA DEMANDA (Histórico + Projeções)
    # =========================================================================
    
    # Linha do Histórico Logístico
    ax.plot(anos_hist, massa_historica, color='black', marker='o', markersize=4,
             linewidth=2.5, label='Historical Demand $\\Phi_{hist}$', zorder=10)
             
    # Linha principal da demanda esperada (E[Φ]) - O trecho futuro
    ax.plot(anos_conexao, media_conexao, color='#228B22', linewidth=2.5, linestyle='-', 
             label='Expected Demand $E[\\Phi]$')
    ax.fill_between(anos_conexao, 0, media_conexao, color='#228B22', alpha=0.2, 
                     label='Expected Demand Zone')
    
    # Adiciona banda de confiança P95 estocástica
    if demand_p95 is not None:
        ax.fill_between(anos_conexao, media_conexao, p95_conexao, color='#ED120E', alpha=0.1,
                         label='Stress-Test Zone ($P_{95}$)')
        ax.plot(anos_conexao, p95_conexao, color="#ED120E", linewidth=1.5, linestyle='--', alpha=0.7,
                 label='Robust Stress-Test $P_{95}$')
    
    # =========================================================================
    # 3. PLOTAGEM DAS CAPACIDADES DE CADA MÉTODO
    # =========================================================================
    name_map = {
        'BASELINE': 'BASELINE',
        'SA': 'SA',
        'GA': 'GA',
    }

    linewidths = {
        'BASELINE': 2,
        'SA': 4,
        'GA': 2
    }
    
    for method, sol in results_dict.items():
        m_name = name_map.get(method, method)
        
        # Capacidade anual nominal
        total_annual_cap = sum(d['Cap'] * 12 for d in sol.open_depots)
        max_processing_cap = total_annual_cap * 1.5  # Limite operacional (Cross-docking)
        
        color = custom_palette.get(m_name, 'orange')
        
        # Desenha Capacidade Nominal atravessando o histórico e futuro para comparar a escala 
        ax.hlines(y=total_annual_cap, xmin=anos_hist[0], xmax=anos_futuros[-1], color=color, linewidth=linewidths.get(m_name, 2), linestyle='-',
                  label=f'Physical Capacity ({m_name})', zorder=4)
        
        # Linha Tracejada: Limite Operacional
        ax.hlines(y=max_processing_cap, xmin=anos_hist[0], xmax=anos_futuros[-1], color=color, linewidth=linewidths.get(m_name, 2), linestyle='--', alpha=0.7, label=f'Operational Limit ({m_name})', zorder=4)
        
        # Encontra ano de saturação
        saturation_year = None
        for i, demand in enumerate(demand_by_year):
            if demand >= total_annual_cap:
                saturation_year = anos_futuros[i]
                break
        
        if saturation_year:
            ax.axvline(x=saturation_year, color=color, linestyle=':', alpha=0.3, linewidth=1)
            ax.text(saturation_year + 0.2, total_annual_cap * 0.5, 
                    f'Sat. {saturation_year}', rotation=90, fontsize=9, alpha=0.9, color=color, fontweight='bold')
    
    # =========================================================================
    # 4. ANOTAÇÕES ADICIONAIS
    # =========================================================================
    if demand_p95 is not None:
        ax.scatter(anos_futuros[-1], demand_p95[-1], color='#ED120E', s=100, zorder=12, marker='D', edgecolor='black')
        
        # Rótulo estilo caixa de texto para não se perder com as linhas do grid
        ax.annotate(f'Robust Design Point\n{demand_p95[-1]/1000:,.0f} tons',
                     xy=(anos_futuros[-1], demand_p95[-1]),
                     xytext=(anos_futuros[-1] - 4, demand_p95[-1] * 1.15),
                     arrowprops=dict(facecolor='black', shrink=0.05, width=1.5, headwidth=6, alpha=0.8),
                     fontsize=9, fontweight='bold', ha='center',
                     bbox=dict(boxstyle="round,pad=0.3", facecolor='white', edgecolor='#ED120E', alpha=0.9), zorder=15)
    
    # =========================================================================
    # 5. ESTÉTICA ACADÊMICA
    # =========================================================================
    ax.set_title('Evolution of Expected Demand vs. Static Network Capacity\n(Historical S-Curve + 15-Year Planning Horizon)', 
              fontsize=14, fontweight='bold', pad=15)
    ax.set_xlabel('Timeline (Years)', fontsize=12)
    ax.set_ylabel('Total Packaging Volume (kg)', fontsize=12)
    
    # Ajustar as "marcas" do eixo X para mostrar os anos a cada 3 anos com rotação
    todos_anos = np.concatenate([anos_hist, anos_futuros])
    ax.set_xticks(np.arange(todos_anos[0], todos_anos[-1] + 1, 3))
    ax.tick_params(axis='x', rotation=45)
    
    # Formatação limpa do eixo Y (mantendo a notação científica da Matplotlib)
    formatter = ticker.ScalarFormatter(useMathText=True)
    formatter.set_scientific(True)
    formatter.set_powerlimits((-2, 4))
    ax.yaxis.set_major_formatter(formatter)
    
    ax.legend(title='Simulation Variables & Configurations', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=10)
    ax.grid(True, ls='--', alpha=0.4)
    
    plt.tight_layout()
    plt.savefig('Capacidade_vs_Demanda_Curva_S.png', dpi=300, bbox_inches='tight')

# ==============================================================================
# 7.4 Função para Salvar Resultados Detalhados em Excel (Incluindo Custos, Rotas, Utilização e Mapas)
# ==============================================================================

def salvar_excel(sol, method_name, map_files):
    fname = f"Resultados_{method_name}.xlsx"
    
    # =========================================================================
    # CÁLCULO AUTOMÁTICO DO INTERVALO DE CONFIANÇA PARA A ABA DE CUSTOS
    # =========================================================================
    custos_list =[]
    
    # Fatores de Ineficiência (Mínimo e Máximo) baseados na Literatura
    gaps = {
        'Routing'  : [0.05, 0.15], # Logística, Tempo, Paradas (5% a 15%)
        'Inventory': [0.10, 0.20], # Estoque (10% a 20%)
        'Penalties': [0.15, 0.30], # Multas (15% a 30%)
        'Fixed'    : [0.00, 0.00]  # CAPEX, OPEX, Frota (0%)
    }
    
    total_opt = 0; total_min = 0; total_max = 0
    
    for comp, val in sol.cost_breakdown.items():
        if method_name == "BASELINE":
            # Aplica os Gaps empíricos apenas no cenário puramente humano
            if comp in['Logistics (E1)', 'Time', 'Stops']:
                g_min, g_max = gaps['Routing']
            elif 'Inventory' in comp:
                g_min, g_max = gaps['Inventory']
            elif 'Penalty' in comp or 'Fine' in comp:
                g_min, g_max = gaps['Penalties']
            else:
                g_min, g_max = gaps['Fixed']
                
            val_min = val * (1.0 + g_min)
            val_max = val * (1.0 + g_max)
        else:
            # Nas metaheurísticas, o valor é estrito (não há erro humano)
            val_min = val; val_max = val
            
        custos_list.append({
            'Componente de Custo': comp,
            'Valor Otimizado (MILP) - R$': val,
            'Estimativa Otimista (Limite Inferior) - R$': val_min,
            'Estimativa Pessimista (Limite Superior) - R$': val_max
        })
        
        total_opt += val; total_min += val_min; total_max += val_max
        
    custos_list.append({
        'Componente de Custo': 'TOTAL LCC (15 Anos)',
        'Valor Otimizado (MILP) - R$': total_opt,
        'Estimativa Otimista (Limite Inferior) - R$': total_min,
        'Estimativa Pessimista (Limite Superior) - R$': total_max
    })
    
    df_custos = pd.DataFrame(custos_list)

    with pd.ExcelWriter(fname, engine='openpyxl') as w:
        df_custos.to_excel(w, sheet_name='Custos', index=False)
        pd.DataFrame(sol.open_depots).to_excel(w, sheet_name='Infraestrutura', index=False)
        pd.DataFrame(list(sol.fleet_counts.items()), columns=['Veículo', 'Qtd']).to_excel(w, sheet_name='Frota Otimizada', index=False)
        pd.DataFrame(sol.facility_utilization).to_excel(w, sheet_name='Ocupacao_Instalacoes', index=False)
        pd.DataFrame(sol.fleet_utilization).to_excel(w, sheet_name='Ocupacao_Frota', index=False)
        
        if sol.mass_balance_e1: 
            pd.DataFrame(sol.mass_balance_e1).to_excel(w, sheet_name='Balanco_Massa', index=False)
        pd.DataFrame(sol.routes).to_excel(w, sheet_name='Rotas E1', index=False)

        if sol.e2_fleet_usage:
            pd.DataFrame(sol.e2_fleet_usage).to_excel(w, sheet_name='Frota_E2_Global', index=False)
        if sol.mass_balance_e2:
            pd.DataFrame(sol.mass_balance_e2).to_excel(w, sheet_name='Despachos_E2_Detalhado', index=False)

        wb = w.book
        ws = wb.create_sheet('Mapas')
        r=1
        for img in map_files:
            if os.path.exists(img): 
                ws.add_image(XLImage(img), f'A{r}')
                r += 40
                
    return fname

# ==============================================================================
# 7.5 Função para Gerar Gráfico de Tempos de Execução por Fase e Algoritmo (Comparativo de Desempenho)
# ==============================================================================

def plot_execution_times(execution_times, save_path='Execution_Times_Chart.png'):
    """Gera gráfico de barras comparando os tempos de execução."""
    
    # Renomeei de 'phases' para 'algorithms' para ficar claro o que estamos iterando
    algorithms = ['BASELINE', 'SA', 'GA']
    phase1_times = []
    phase2_times = []
    phase3_times = []
    
    for algo in algorithms:
        t1 = execution_times['phases'].get('Phase 1 - Strategic Design', {}).get(algo, 0)
        t2 = execution_times['phases'].get('Phase 2 - Tactical IRP', {}).get(algo, 0)
        t3 = execution_times['phases'].get('Phase 3 - Vehicle Assignment', {}).get(algo, 0)
        phase1_times.append(t1)
        phase2_times.append(t2)
        phase3_times.append(t3)
    
    x = np.arange(len(algorithms))
    width = 0.25
    
    fig, ax = plt.subplots(figsize=(12, 6))

    custom_palette = {
        'BASELINE': '#FF8C00', # Laranja
        'SA': "#1F77B4",       # Azul Escuro
        'GA': "#2CA02C"        # Verde Água
    }

    def get_shades(hex_color):
        """Retorna tons claro, médio e escuro calculados a partir da cor base."""
        c = np.array(mcolors.to_rgb(hex_color))
        light = c + (1 - c) * 0.45  # Mistura com 45% de branco
        medium = c                  # Cor base (original)
        dark = c * 0.6              # Escurece a cor em 40% (mistura com preto)
        return light, medium, dark

    # Listas para armazenar as cores de cada fase baseada na metaheurística
    colors_p1, colors_p2, colors_p3 = [], [], []
    for algo in algorithms:
        l, m, d = get_shades(custom_palette[algo])
        colors_p1.append(l)
        colors_p2.append(m)
        colors_p3.append(d)
    
    # Plotagem usando as listas de cores (adicionado edgecolor para destacar as barras)
    bars1 = ax.bar(x - width, phase1_times, width, color=colors_p1, edgecolor='black', linewidth=0.7)
    bars2 = ax.bar(x, phase2_times, width, color=colors_p2, edgecolor='black', linewidth=0.7)
    bars3 = ax.bar(x + width, phase3_times, width, color=colors_p3, edgecolor='black', linewidth=0.7)
    
    ax.set_xlabel('Algorithm')
    ax.set_ylabel('Time (seconds)')
    ax.set_title('Execution Time Comparison by Phase')
    ax.set_xticks(x)
    ax.set_xticklabels(algorithms)

    # Formatação limpa do eixo Y (mantendo a notação científica da Matplotlib)
    formatter = ticker.ScalarFormatter(useMathText=True)
    formatter.set_scientific(True)
    formatter.set_powerlimits((-2, 4))
    ax.yaxis.set_major_formatter(formatter)
    
    # Legenda customizada demonstrando o conceito dos tons
    legend_elements = [
        Patch(facecolor='#E0E0E0', edgecolor='black', label='Phase 1 - Strategic Design (Light)'),
        Patch(facecolor='#888888', edgecolor='black', label='Phase 2 - Tactical IRP (Medium)'),
        Patch(facecolor='#333333', edgecolor='black', label='Phase 3 - Vehicle Assignment (Dark)')
    ]
    ax.legend(handles=legend_elements, title='Framework Phase', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=10)
    
    ax.grid(True, alpha=0.3)
    
    # Adicionar valores nas barras
    for bars in [bars1, bars2, bars3]:
        for bar in bars:
            height = bar.get_height()
            if height > 0:
                ax.annotate(f'{height:.1f}',
                           xy=(bar.get_x() + bar.get_width() / 2, height),
                           xytext=(0, 3),
                           textcoords="offset points",
                           ha='center', va='bottom', fontsize=8)
    
    plt.yscale('log')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')

# ==============================================================================
# 8. MAIN EXECUTION
# ==============================================================================

def format_time(seconds):
    """Formata o tempo em segundos para formato legível."""
    if seconds < 60:
        return f"{seconds:.2f} seconds"
    elif seconds < 3600:
        return f"{seconds/60:.2f} minutes"
    else:
        return f"{seconds/3600:.2f} hours"

def log_time_breakdown(phase_name, subphase_name, elapsed_time, log_dict):
    """
    Registra o tempo gasto em uma fase específica.
    Modifica o dicionário in-place e também o retorna para compatibilidade.
    """
    if phase_name not in log_dict:
        log_dict[phase_name] = {}
    log_dict[phase_name][subphase_name] = elapsed_time
    
    print(f"   ⏱️  {subphase_name}: {format_time(elapsed_time)}")
    return log_dict  # Retorna para permitir encadeamento

def run_full_execution():
    print(f"\n{'='*60}\nOTIMIZAÇÃO TWO-ECHELON LIRP (V31.7)\n{'='*60}")
    # print(f"\n{'='*60}\nCenário Realista\n{'='*60}")
    
    # =========================================================================
    # Dicionário para armazenar tempos de execução
    # =========================================================================
    execution_times = {
        'total_start': time.time(),
        'phases': {}
    }

    instance = '25_G38.xlsx'
    cenario_base = 'Realista'
    
    data = LRPData(instance, cenario_base)
    
    # =========================================================================
    # PARÂMETROS DE EXECUÇÃO
    # =========================================================================
    velocidade = False  # Define se a execução será em modo rápido (True) ou completo (False)

    if velocidade:
        print("\n>> Modo VELOCIDADE ATIVADO: Configurações reduzidas para execução rápida (para testes e depuração).")
        t_calib = 3000      # Tempo para cada execução na calibração
        reps =       2      # Repetições para cada configuração na calibração

        t_metah = 3000      # Tempo Fase 1 (Metaheurística)
        E_MAX =  50000      # Nº Máximo de Avaliações da FO (Fase 1)

        t_milp1 = 1200      # Tempo Fase 2 (MILP - Roteamento)
        t_milp2 = 1000      # Tempo Fase 3 (MILP - Alocação)
    else:
        print("\n>> Modo VELOCIDADE DESATIVADO: Configurações completas para execução robusta (para experimentos finais).")
        t_calib = 12000     # Tempo para cada execução na calibração
        reps =       30     # Repetições para cada configuração na calibração

        t_metah = 12000     # Tempo Fase 1 (Metaheurística)
        E_MAX =  100000     # Nº Máximo de Avaliações da FO (Fase 1)

        t_milp1 =  3600     # Tempo Fase 2 (MILP - Roteamento)
        t_milp2 =  3600     # Tempo Fase 3 (MILP - Alocação)
    
    eval1 = Phase1Evaluator(data)
        
    best_params = {
        'SA' : {'p0_start': 0.5, 'alpha': 0.9},
        'GA' : {'pop_size': 30, 'cx_rate': 0.7, 'mut_rate': 0.2}
    } # Meta_Testing_6.7_New
    
    # =========================================================================
    # FASE 1: STRATEGIC NETWORK DESIGN
    # =========================================================================
    print(f"\n{'='*60}")
    print("FASE 1: STRATEGIC NETWORK DESIGN")
    print(f"{'='*60}")
    
    phase1_times = {}
    final_depots = {}
    final_fleets = {}
    
    # 1.1 Simulated Annealing (SA)
    print("\n>> Executando SA (Fase 1)...")
    sa_start = time.time()
    sa_dep, sa_flt, sa_h = simulated_annealing(data, eval1, max_time=t_metah, max_evals=E_MAX, 
                                                **best_params['SA'])
    sa_time = time.time() - sa_start
    phase1_times['SA'] = sa_time
    final_depots['SA'] = sa_dep
    final_fleets['SA'] = sa_flt
    execution_times['phases'] = log_time_breakdown('Phase 1 - Strategic Design', 'SA', 
                                          sa_time, execution_times['phases'])
    
    # 1.2 Genetic Algorithm (GA)
    print("\n>> Executando GA (Fase 1)...")
    ga_start = time.time()
    ga_dep, ga_flt, ga_h = genetic_algorithm(data, eval1, max_time=t_metah, max_evals=E_MAX,
                                              **best_params['GA'])
    ga_time = time.time() - ga_start
    phase1_times['GA'] = ga_time
    final_depots['GA'] = ga_dep
    final_fleets['GA'] = ga_flt
    execution_times['phases'] = log_time_breakdown('Phase 1 - Strategic Design', 'GA', 
                                          ga_time, execution_times['phases'])
    
    # =========================================================================
    # GRÁFICO DE CONVERGÊNCIA FASE 1 (Comparativo de Desempenho e Tempo)
    # =========================================================================

    sa_x, sa_y = zip(*sa_h)
    ga_x, ga_y = zip(*ga_h)
    
    plt.figure(figsize=(10, 6))

    custom_palette = {
        'BASELINE': '#FF8C00',
        'SA': "#1F77B4",
        'GA': "#2CA02C"
    }

    plt.plot(sa_x, sa_y, label=f'SA ({format_time(sa_time)})', color=custom_palette['SA'])
    plt.plot(ga_x, ga_y, label=f'GA ({format_time(ga_time)})', color=custom_palette['GA'])

    # Notação científica exponencial para o eixo Y
    formatter = ticker.ScalarFormatter(useMathText=True)
    formatter.set_scientific(True)
    formatter.set_powerlimits((-2, 4))
    plt.gca().yaxis.set_major_formatter(formatter)

    # Notação científica para o eixo X (avaliações)
    plt.gca().xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

    plt.yscale('log')
    plt.title('Phase 1 Convergence - Time Comparison')
    plt.xlabel('Function Evaluations')
    plt.ylabel('Cost')
    plt.legend()
    plt.savefig('Convergencia_Fase1.png', dpi=300, bbox_inches='tight')
    
    # =========================================================================
    # FASE 2 e 3: Para cada método (incluindo Baseline)
    # =========================================================================
    files_to_zip = ['Convergencia_Fase1.png']
    final_results = {}
    
    # Lista de configurações a avaliar
    configurations = []
    
    configurations.append(('BASELINE', data.get_baseline_depots()))
    
    configurations.extend([
        ('SA', final_depots['SA']),
        ('GA', final_depots['GA'])
    ])
    
    for name, depot_config in configurations:
        print(f"\n{'-'*60}")
        print(f">>> Avaliando Rede {name}...")
        print(f"{'-'*60}")
        
        # =====================================================================
        # FASE 2: TACTICAL 2E-IRP (SET PARTITIONING)
        # =====================================================================
        print(f"\n{'='*60}")
        print(f"FASE 2: TACTICAL 2E-IRP - {name}")
        print(f"{'='*60}")
        
        phase2_start = time.time()
        milp_res = solve_phase2_milp(data, depot_config, time_limit=t_milp1)
        phase2_time = time.time() - phase2_start
        execution_times['phases'] = log_time_breakdown('Phase 2 - Tactical IRP', f'{name}', 
                                              phase2_time, execution_times['phases'])
        
        if not milp_res:
            print(f"   ⚠️  {name}: MILP Fase 2 não convergiu. Pulando...")
            continue
        
        # =====================================================================
        # FASE 3: OPERATIONAL VEHICLE SCHEDULING (ASSIGNMENT)
        # =====================================================================
        print(f"\n{'='*60}")
        print(f"FASE 3: OPERATIONAL VEHICLE SCHEDULING - {name}")
        print(f"{'='*60}")
        
        phase3_start = time.time()
        sol = build_final_solution(data, depot_config, milp_res, time_limit=t_milp2)
        phase3_time = time.time() - phase3_start
        execution_times['phases'] = log_time_breakdown('Phase 3 - Vehicle Assignment', f'{name}', 
                                              phase3_time, execution_times['phases'])
        
        # Geração de mapas e salvamento
        maps = gerar_mapa_poligonal(sol, data, name)
        excel_file = salvar_excel(sol, name, maps)
        files_to_zip.extend(maps + [excel_file])
        final_results[name] = sol
        
        print(f"\n   ✅ [{name}] Custo Final: R$ {sol.total_cost:,.2f}")
        print(f"   📊 Tempo total {name}: {format_time(phase2_time + phase3_time)}")

        # --- LIMPEZA PROFUNDA APÓS AVALIAR UM ALGORITMO INTEIRO ---
        gc.collect()
        if HAS_GUROBI:
            gp.disposeDefaultEnv()
        # ----------------------------------------------------------
    
    # =========================================================================
    # GRÁFICOS FINAIS E RELATÓRIO DE TEMPOS
    # =========================================================================
    if final_results:
        plot_cost_comparison_v3(final_results)
        files_to_zip.append('Comparativo_Custos_V3.png')
        
        plot_capacity_vs_demand(final_results, data)
        files_to_zip.append('Capacidade_vs_Demanda_Estocastico.png')

        # NOVO: Gráfico de tempos de execução
        plot_execution_times(execution_times, 'Execution_Times_Chart.png')
        files_to_zip.append('Execution_Times_Chart.png')
    
    # =========================================================================
    # RELATÓRIO DE TEMPOS DE EXECUÇÃO
    # =========================================================================
    execution_times['total_end'] = time.time()
    total_time = execution_times['total_end'] - execution_times['total_start']
    execution_times['total_elapsed'] = total_time
    
    print(f"\n{'='*70}")
    print("RELATÓRIO DE TEMPOS DE EXECUÇÃO")
    print(f"{'='*70}")
    
    print(f"\n📊 RESUMO POR FASE:")
    print(f"{'-'*50}")
    
    total_by_phase = {}
    for phase, subphases in execution_times['phases'].items():
        phase_total = sum(subphases.values())
        total_by_phase[phase] = phase_total
        print(f"\n{phase}:")
        for sub, t in subphases.items():
            print(f"   ├─ {sub}: {format_time(t)}")
        print(f"   └─ TOTAL {phase}: {format_time(phase_total)}")
    
    print(f"\n📈 RESUMO GERAL:")
    print(f"{'-'*50}")
    print(f"   Tempo total de execução: {format_time(total_time)}")
    
    # Tabela comparativa de tempos por algoritmo
    print(f"\n📊 COMPARAÇÃO ENTRE ALGORITMOS (Phase 1 + Phase 2 + Phase 3):")
    print(f"{'-'*60}")
    print(f"{'Algoritmo':<15} {'Phase 1':<15} {'Phase 2':<15} {'Phase 3':<15} {'TOTAL':<15}")
    print(f"{'-'*60}")
    
    for algo in ['SA', 'GA']:
        t1 = phase1_times.get(algo, 0)
        t2 = execution_times['phases'].get('Phase 2 - Tactical IRP', {}).get(algo, 0)
        t3 = execution_times['phases'].get('Phase 3 - Vehicle Assignment', {}).get(algo, 0)
        total_algo = t1 + t2 + t3
        print(f"{algo:<15} {format_time(t1):<15} {format_time(t2):<15} {format_time(t3):<15} {format_time(total_algo):<15}")
    
    if 'BASELINE' in final_results:
        t2_base = execution_times['phases'].get('Phase 2 - Tactical IRP', {}).get('BASELINE', 0)
        t3_base = execution_times['phases'].get('Phase 3 - Vehicle Assignment', {}).get('BASELINE', 0)
        print(f"{'BASELINE':<15} {'N/A':<15} {format_time(t2_base):<15} {format_time(t3_base):<15} {format_time(t2_base + t3_base):<15}")
    
    # Salvar relatório de tempos em arquivo
    report_path = "Relatorio_Tempos_Execucao.txt"
    with open(report_path, 'w', encoding='utf-8') as f:
        f.write("="*70 + "\n")
        f.write("RELATÓRIO DE TEMPOS DE EXECUÇÃO - MP-HF-SD-2E-LIRP\n")
        f.write(f"Data da execução: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write("="*70 + "\n\n")
        
        f.write("TEMPOS POR FASE:\n")
        f.write("-"*50 + "\n")
        for phase, subphases in execution_times['phases'].items():
            f.write(f"\n{phase}:\n")
            for sub, t in subphases.items():
                f.write(f"  - {sub}: {t:.2f} segundos ({format_time(t)})\n")
        
        f.write(f"\nTEMPO TOTAL: {total_time:.2f} segundos ({format_time(total_time)})\n")
        
        f.write("\n\nCOMPARAÇÃO ENTRE ALGORITMOS:\n")
        f.write("-"*60 + "\n")
        f.write(f"{'Algoritmo':<15} {'Phase 1 (s)':<12} {'Phase 2 (s)':<12} {'Phase 3 (s)':<12} {'TOTAL (s)':<12}\n")
        f.write("-"*60 + "\n")
        
        for algo in ['SA', 'GA']:
            t1 = phase1_times.get(algo, 0)
            t2 = execution_times['phases'].get('Phase 2 - Tactical IRP', {}).get(algo, 0)
            t3 = execution_times['phases'].get('Phase 3 - Vehicle Assignment', {}).get(algo, 0)
            f.write(f"{algo:<15} {t1:<12.2f} {t2:<12.2f} {t3:<12.2f} {t1+t2+t3:<12.2f}\n")
        
        if 'BASELINE' in final_results:
            t2 = execution_times['phases'].get('Phase 2 - Tactical IRP', {}).get('BASELINE', 0)
            t3 = execution_times['phases'].get('Phase 3 - Vehicle Assignment', {}).get('BASELINE', 0)
            f.write(f"{'BASELINE':<15} {'N/A':<12} {t2:<12.2f} {t3:<12.2f} {t2+t3:<12.2f}\n")
    
    files_to_zip.append(report_path)
    print(f"\n📄 Relatório de tempos salvo em: {report_path}")
    
    # =========================================================================
    # COMPACTAR RESULTADOS
    # =========================================================================
    print("\n📦 Compactando resultados...")
    import zipfile
    zname = f"Resultados_2ELIRP_{time.strftime('%Y%m%d_%H%M%S')}.zip"
    with zipfile.ZipFile(zname, 'w') as zf:
        for f in set(files_to_zip):
            if os.path.exists(f):
                zf.write(f)
                # print(f"   Adicionado: {f}")
    
    print(f"\n{'='*60}")
    print("✅ EXECUÇÃO CONCLUÍDA COM SUCESSO!")
    print(f"{'='*60}")
    print(f"📁 Arquivo de resultados: {zname}")
    print(f"⏱️  Tempo total: {format_time(total_time)}")
    print(f"{'='*60}")
    
    return final_results, execution_times

if __name__ == "__main__":
    run_full_execution()
